<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/notebooks/stage_04_feature_engineering/stage_04_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_04_feature_engineering**

## **Introducción**

Esta notebook corresponde al stage_04a - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [6]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [7]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [8]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

  Preparing metadata (setup.py) ... done
Librería instalada: technical-analysis


### 0.3. Importación de librerías


In [9]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos
import ta
import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal

from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

### 0.4. Definición de rutas



In [10]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [11]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/processed/mnq_intraday_labeled.parquet"))
IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [12]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

### 0.5. Códigos auxiliares para carga de datos y visualización


In [13]:
def load_data():

    # Definir la URL del archivo Parquet en Drive
    data_path = f'{drive_path}/data/processed/mnq_intraday_labeled.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [14]:
def info_dataset(df):
  print("Información del dataset:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}")

In [15]:
# Carga del JSON
with IN_ARTIFACT.open("r") as f:
    target_definition_summary = json.load(f)

In [16]:
def print_summary_console(summary: Dict[str, Any]) -> None:
    """Consola legible (sin depender de pandas display)."""
    print("\n" + "=" * 70)
    print(f"STAGE: {summary.get('stage')}")
    print(f"CREATED_AT_UTC: {summary.get('created_at_utc')}")
    print(f"VERSION: {summary.get('version')}")
    print("-" * 70)

    paths = summary.get("paths", {})
    print("[PATHS]")
    for group in ("inputs", "outputs", "reports"):
        print(f"  {group}:")
        for k, v in paths.get(group, {}).items():
            print(f"    - {k}: {v}")

    params = summary.get("params", {})
    print("\n[PARAMS]")
    for k, v in params.items():
        print(f"  - {k}: {v}")

    metrics = summary.get("metrics", {})
    print("\n[METRICS]")
    for k, v in metrics.items():
        print(f"  - {k}: {v}")

    details = summary.get("details", {})
    gw = details.get("gestation_window", {})
    if gw:
        print("\n[GESTATION WINDOW]")
        print(f"  - start: {gw.get('start_hhmm')}")
        print(f"  - end  : {gw.get('end_hhmm')}")
        print(f"  - top_n: {gw.get('top_n')}")
        print(f"  - minute_of_day_min: {gw.get('minute_of_day_min')}")
        print(f"  - minute_of_day_max: {gw.get('minute_of_day_max')}")

    print("=" * 70 + "\n")

In [17]:
#Carga de dataset base:
mnq_intraday_labeled = load_data()
info_dataset(mnq_intraday_labeled)

# Eliminación de filas con NaN
mnq_intraday_labeled = mnq_intraday_labeled.dropna()

# Verificación posterior
info_dataset(mnq_intraday_labeled)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York
Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio 06:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York


In [18]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [19]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

## **1. Relación entre indicadores técnicos, information coefficient y los targets definidos del stage_03**

### **1.1. Indicadores técnicos**

Los indicadores técnicos se construyen a partir de la serie de precios intradía, y en particular sobre la variable de cierre (close). Estas transformaciones matemáticas buscan capturar propiedades dinámicas del mercado tales como tendencia, momentum, reversión, volatilidad y estructura temporal del movimiento de precios.

En el marco actual del proyecto, la variable objetivo (target) ya no se define como un retorno normalizado, sino como un movimiento futuro absoluto en puntos (delta en puntos), medido sobre horizontes temporales discretos (60 y 90 minutos). A partir de estos deltas se definen distintos umbrales económicamente relevantes (base, operativo y cola), que dan lugar a señales de trade y targets binarios asociados. Los valores de delta definidos en el stage_03_target_definition son los siguientes:

In [20]:
print_summary_console(target_definition_summary)


STAGE: stage_03b_target_definition
CREATED_AT_UTC: 2026-01-16T03:29:54.712920+00:00
VERSION: 1.0
----------------------------------------------------------------------
[PATHS]
  inputs:
    - intraday_parquet: data/processed/mnq_intraday.parquet
    - stage03a_summary: reports/stage_03a_target_investigation_summary.json
  outputs:
    - mnq_intraday_labeled: data/processed/mnq_intraday_labeled.parquet
  reports:
    - summary: reports/stage_03b_target_definition_summary.json

[PARAMS]
  - date_col: date
  - close_col: close
  - horizons: [60, 90]
  - drop_na_targets: True
  - top_persist_n: 25

[METRICS]
  - n_rows: 626743
  - n_cols: 15
  - n_days: 1303
  - total_nans: 0
  - h60_n_trade: 160670
  - h60_n_target_op: 73450
  - h60_n_target_tail: 20943
  - h90_n_trade: 180453
  - h90_n_target_op: 86779
  - h90_n_target_tail: 25032

[GESTATION WINDOW]
  - start: 08:21
  - end  : 08:49
  - top_n: 25
  - minute_of_day_min: 501
  - minute_of_day_max: 529



Esto implica que, aunque los indicadores técnicos se calculen directamente sobre el precio, su evaluación no se orienta a explicar la evolución instantánea del close, sino a medir su capacidad para anticipar movimientos futuros de magnitud suficiente, expresados en puntos, dentro de un horizonte temporal determinado.

En consecuencia, el vínculo entre indicadores y targets se establece en términos de poder predictivo sobre la ocurrencia de deltas futuros significativos, es decir, sobre la probabilidad de que el mercado alcance determinados umbrales de movimiento (Δ base, Δ operativo o Δ cola) en el horizonte considerado. El análisis posterior se centra, por tanto, en identificar qué indicadores y configuraciones temporales contienen información relevante para discriminar contextos de no-trade, trade operativo o eventos de cola, coherentes con los targets definidos en el stage_03.

### **1.2. Information Coefficient (IC) versus targets**

En el dataset `mnq_intraday_labeled`, cada fila representa una decisión potencial en un instante $𝑡$ del intradía. Para ese instante, los targets (trade_60, target_op_60, target_tail_60, y sus equivalentes a 90 minutos) indican si, partiendo desde $𝑡$, el precio alcanza o no determinados umbrales de movimiento en un horizonte futuro fijo.

La ventana de gestión (08:20–08:40) no redefine el target ni introduce una nueva variable temporal, sino que delimita el conjunto de instantes $𝑡$
en los cuales el sistema está habilitado a evaluar señales y tomar decisiones. En consecuencia, el análisis se restringe exclusivamente a las filas cuyo timestamp pertenece a dicha ventana horaria.

Por su parte, la llamada ventana de ejecución (por ejemplo, 09:10–09:40 para un horizonte de 60 minutos) no es una entidad explícita del modelo ni del cálculo del IC. Esta ventana surge de manera implícita, ya que corresponde al intervalo temporal donde se materializa el resultado futuro de las decisiones tomadas en la ventana de gestión. Es decir, para cada $𝑡 ∈ [08:10,08:40]$, el target a 60 minutos refleja el comportamiento del precio en $𝑡+60$, que naturalmente cae dentro de esa franja posterior.

Bajo este esquema, el Information Coefficient (IC) se calcula correlacionando, para cada instante $𝑡$ de la ventana de gestión:

$𝑋_{t}$ : el valor del indicador técnico calculado con información disponible hasta $𝑡$

$𝑌_{t,h}$: el target asociado a ese mismo instante 𝑡 y a un horizonte
ℎ (por ejemplo, `delta_pts_60` o `trade_60`).

De este modo, el IC mide directamente la capacidad del indicador, evaluado en el momento de decisión, para anticipar la ocurrencia de un movimiento futuro económicamente relevante. No se comparan ventanas horarias entre sí, ni se agregan bloques temporales: la relación se establece fila a fila, respetando estrictamente la causalidad temporal.

En términos operativos, un IC positivo indica que ciertos estados del mercado, caracterizados por los indicadores técnicos en la ventana de gestión, están sistemáticamente asociados a una mayor probabilidad de alcanzar los targets definidos en el stage_03. Esto justifica su uso como variables explicativas en el entrenamiento del modelo predictivo.

### **1.3. Alineación entre el punto 8 (stage_03b) y el cálculo del IC (stage_04)**

La construcción de las ventanas operativas desarrollada en el punto 8 del stage_03b establece una separación conceptual fundamental entre:

- una ventana de gestación (predicción), donde se origina la información anticipatoria, y

- una ventana de expansión (ejecución), donde los movimientos alcanzan magnitud económica explotable.

Esta separación no entra en conflicto con la definición de los targets ni con el cálculo del Information Coefficient (IC); por el contrario, ambos enfoques son complementarios y coherentes, siempre que se entienda correctamente el rol temporal de cada elemento.



#### **1.3.1. Nivel estadístico (dataset y targets)**


En `mnq_intraday_labeled`, los targets (`delta_pts_h`, `trade_h`, `target_op_h`, `target_tail_h`) están definidos fila a fila, para cada instante t, como el resultado del movimiento futuro observado en t+h.

Esto significa que:
- el target está anclado temporalmente al instante t
- el horizonte h determina cuándo se materializa el resultado,
- no existen targets definidos “por ventana”, sino por decisión potencial en un minuto específico.

Cuando el análisis se restringe a la ventana de gestación (por ejemplo, 08:20-08:40), lo que se hace es seleccionar el subconjunto de instantes
t que, según el análisis empírico del punto 8, concentran información anticipatoria relevante.

En este contexto, el IC mide:
 - La relación estadística entre el estado del mercado en t (capturado por los indicadores técnicos) y el resultado futuro asociado a ese mismo t, que se materializa en la ventana de expansión.

#### **1.3.2. Nivel operativo (modelo y ejecución)**

Desde el punto de vista operativo, el esquema temporal se interpreta de la siguiente manera:

- Ventana de gestación (08:20-08:40)
  - Se calculan los indicadores técnicos.
  - El modelo evalúa si, desde esos instantes, es probable alcanzar un delta relevante a 60 o 90 minutos.
  - Aquí reside la capacidad predictiva.

- Ventana de expansión (09:10-09:40)
  - Es el período donde, empíricamente, los movimientos alcanzan mayor frecuencia y magnitud.
  - Las predicciones generadas previamente habilitan (o no) la toma de operaciones reales.
  - El punto de entrada puede ubicarse en cualquier minuto de esta franja, sujeto a reglas operativas adicionales.

- Horizonte de resultado
- El cierre de la operación ocurre según:
- `delta_op` (objetivo operativo),
- `delta_tail` (extensión),
- o reglas de stop,
  
  siempre respetando el horizonte temporal definido desde el instante de entrada.

#### **1.3.3. Rol del IC dentro de este esquema**

El **Information Coefficient** no evalúa la ejecución, sino la calidad de la información generada en la ventana de gestación.

Su función es responder a la pregunta:

  ¿Los indicadores técnicos calculados en la ventana de gestación contienen información útil para anticipar los movimientos que se expanden y se monetizan más adelante?

Por lo tanto:
- El IC se calcula exclusivamente en la ventana de gestación.
- Los targets ya incorporan, de forma implícita, el desfase temporal hacia la ventana de expansión.
- La coherencia temporal y la ausencia de data leakage quedan garantizadas por construcción.

### **1.4. Conclusión sintética**

La lógica del point 8 define dónde nace la información y dónde se ejecuta la operación.

El cálculo del IC, en el stage_04, cuantifica qué tan informativa es esa ventana de gestación respecto de los resultados que se materializan posteriormente.

Ambos enfoques describen el mismo fenómeno desde niveles distintos (operativo vs estadístico) y están plenamente alineados dentro del diseño del pipeline.

## **2. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

### 2.1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [21]:
def calcular_rsi(df=mnq_intraday_labeled, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

### 2.2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [22]:
def calcular_momentum(df=mnq_intraday_labeled, target='close' ):
  momentum_columns = ['momentum_10', 'momentum_5','momentum_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['momentum_10'] = grupo[target].pct_change(10)
        grupo['momentum_5'] = grupo[target].pct_change(5)
        grupo['momentum_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

### 2.3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [23]:
def calcular_volumen_ratio(df=mnq_intraday_labeled, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

### 2.4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [24]:
def calcular_macd(df=mnq_intraday_labeled, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

### 2.5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [25]:
def calcular_ema(df=mnq_intraday_labeled, target='close'):
    ema_columns = ['price_ema15', 'price_ema20', 'price_ema30',  'price_ema60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['price_ema15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['price_ema20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['price_ema30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['price_ema60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

### 2.6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [26]:
def calcular_stochastic(df=mnq_intraday_labeled, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


### 2.7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [27]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday_labeled, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [28]:
def calcular_bollinger_resume(df=mnq_intraday_labeled, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

### 2.8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [29]:
def calcular_atr(df=mnq_intraday_labeled, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

In [30]:
def calcular_atr_14(df=mnq_intraday_labeled, target='close'):
    atr_columns = ['atr_norm']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=14)
        grupo['atr'] = atr.average_true_range()
        grupo['atr_norm'] = grupo['atr'] / grupo[target]
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, atr_columns

### 2.9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [31]:
def calcular_roc(df=mnq_intraday_labeled, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

### 2.10. **Cálculo final de indicadores técnicos**

Calculamos los indicadores técnicos

In [32]:
mnq_intraday_with_indicators = mnq_intraday_labeled.copy()

mnq_intraday_with_indicators, rsi_columns = calcular_rsi(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, momentum_columns = calcular_momentum(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, volume_ratio_columns = calcular_volumen_ratio(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, macd_columns = calcular_macd(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, ema_columns = calcular_ema(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, stoch_columns = calcular_stochastic(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, bollinger_columns = calcular_bollinger(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, atr_columns = calcular_atr(mnq_intraday_with_indicators)
mnq_intraday_with_indicators, roc_columns = calcular_roc(mnq_intraday_with_indicators)

Construimos el listado de indicadores técnicos

In [33]:
indicator_columns = (
    rsi_columns
    + momentum_columns
    + volume_ratio_columns
    + macd_columns
    + ema_columns
    + stoch_columns
    + bollinger_columns
    + atr_columns
    + roc_columns
)

Filtramos todos los NaNs del dataset

In [34]:
info_dataset(mnq_intraday_with_indicators)

# Eliminación de filas con NaN
mnq_intraday_with_indicators = mnq_intraday_with_indicators.dropna()

# Verificación posterior
info_dataset(mnq_intraday_with_indicators)

Información del dataset:

	Cantidad de días: 1303
	Registros por día: 481
	Hora diaria de inicio 06:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York
Información del dataset:

	Cantidad de días: 1303
	Registros por día: 392
	Hora diaria de inicio 07:59
	Hora diaria de final 14:30
	Zona horaria: America/New_York


In [35]:
assert not mnq_intraday_with_indicators.isna().any().any(), \
    "El dataset contiene NaN"

##**3. Calculo de Information Coefficient (IC)**

### **3.1. Tipos de targets y criterios para el cálculo del Information Coefficient (IC)**

En el dataset `mnq_intraday_labeled` coexisten distintos tipos de variables objetivo, que responden a naturalezas estadísticas y operativas diferentes. En consecuencia, el cálculo e interpretación del Information Coefficient (IC) debe adaptarse a cada caso, manteniendo un criterio metodológico coherente.

#### **3.1.1. Targets continuos `delta_pts_h`**

Las variables delta_pts_h representan el movimiento futuro absoluto en puntos, medido desde un instante 𝑡 hasta 𝑡+ℎ. Este tipo de target constituye el caso más directo y conceptualmente “puro” para el cálculo del IC.

En este contexto, el IC mide si un indicador técnico es capaz de ordenar correctamente la dirección y la magnitud relativa de los movimientos futuros. El coeficiente recomendado es la correlación de Spearman, ya que:
- no asume relaciones lineales,
- es robusta frente a valores extremos,
- evalúa asociaciones monotónicas, más adecuadas para series financieras.

La interpretación es directa:

- un IC positivo indica que valores más altos del indicador tienden a asociarse con deltas futuros mayores,
- un IC negativo indica la relación inversa.

Por estas razones, `delta_pts_h` se adopta como target principal para el análisis de IC.

#### **3.1.2. Targets discretos ordinales: `trade_h` (-1, 0, +1)**

La variable `trade_h` codifica el sentido operativo del movimiento futuro, distinguiendo entre posiciones cortas (-1), ausencia de trade (0) y posiciones largas (+1). Se trata de un target ordinal, no continuo.

En este caso, el IC evalúa si el indicador tiende a tomar valores sistemáticamente mayores o menores a medida que el sentido del trade progresa desde short hacia long. Nuevamente, la correlación de Spearman resulta apropiada, ya que respeta el orden implícito de las categorías.

La interpretación es la siguiente:

- un IC positivo indica que el indicador suele ser mayor en escenarios long que en no-trade y short,
- un IC negativo indica el patrón opuesto.

Dado que el estado “no-trade” suele ser dominante en frecuencia, este target puede generar ICs atenuados. Por este motivo, el análisis puede complementarse con evaluaciones restringidas al subconjunto de observaciones donde `trade_h ≠ 0`, con el fin de aislar el componente puramente direccional.

#### **3.1.3. Targets binarios: `target_op_h` y `target_tail_h` (0/1)**


Las variables `target_op_h` y `target_tail_h` indican la ocurrencia o no de eventos económicamente relevantes (alcance del objetivo operativo o de cola). Se trata de targets binarios, asociados a eventos discretos.

En este contexto, el IC mide si el indicador técnico tiende a tomar valores más altos (o más bajos) cuando el evento ocurre (1) frente a cuando no ocurre (0). Aunque el target es binario, la correlación de Spearman sigue siendo válida como medida de asociación monotónica.

La interpretación es:

- IC positivo: valores altos del indicador se asocian a mayor probabilidad de alcanzar el objetivo,
- IC negativo: valores altos del indicador se asocian a menor probabilidad del evento.

Dado que estos eventos pueden ser poco frecuentes —especialmente en el caso de `target_tail_h`—, el IC puede presentar mayor variabilidad. Por ello, se recomienda calcular el IC por día y luego resumirlo mediante estadísticas agregadas (media, mediana y dispersión).

#### **3.1.4. Recomendación operativa**


Para cada indicador técnico y cada horizonte ℎ, el análisis de IC se estructura en tres niveles complementarios:

- IC principal: asociación entre el indicador y `delta_pts_h`.
- IC direccional: asociación entre el indicador y `trade_h`.
- IC de evento: asociación entre el indicador y `target_op_h` y `target_tail_h`.

Este enfoque permite evaluar, de manera consistente, la relación entre los indicadores técnicos y la magnitud, dirección y relevancia económica de los movimientos futuros, respetando la estructura temporal y operativa definida en las etapas previas del pipeline.

Si bien el marco operativo define múltiples targets derivados (direccionales y de evento), en este stage el análisis de Information Coefficient se realiza únicamente sobre `delta_pts_h`, por ser la variable base continua a partir de la cual se construyen los demás objetivos. Esto evita redundancias y permite evaluar de forma más limpia la capacidad predictiva de los indicadores técnicos.

### **3.2. Implementación de cálculo de IC**

#### **3.2.1. Código**

In [36]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

#=========================
#Utilidades base
#=========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """
    Calcula el Information Coefficient (IC) usando correlación de Spearman.

    - x: indicador técnico en el instante t
    - y: target asociado a ese mismo t (delta, trade, evento)
    - Ignora valores NaN en ambas series
    - Devuelve np.nan si no hay suficientes observaciones
    """

    # Máscara para quedarnos solo con pares (x, y) válidos
    mask = x.notna() & y.notna()

    # Si hay menos de 3 puntos válidos, la correlación no es confiable
    if mask.sum() < 3:
        return np.nan

    # Correlación de Spearman (rank correlation)
    return spearmanr(x[mask], y[mask]).correlation

def filter_time_window(
    df: pd.DataFrame,
    start="08:10",
    end="08:50"
) -> pd.DataFrame:
    """
    Filtra el DataFrame por una ventana horaria intradía.

    - Se utiliza para aislar la ventana de gestación (predicción)
    - Requiere que el índice del DataFrame sea DatetimeIndex
    """
    return df.between_time(start, end)

def daily_ic(
    df: pd.DataFrame,
    indicator_col: str,
    target_col: str
) -> pd.Series:
    """
    Calcula el IC de Spearman por día.

    - Agrupa el dataset por la columna 'date'
    - Dentro de cada día, calcula IC(indicador, target)
    - Devuelve una serie con un IC por jornada
    """
    return df.groupby("date").apply(
        lambda g: spearman_ic(g[indicator_col], g[target_col])
    )

#=========================
#Tabla IC (indicadores × horizontes)
#=========================

def compute_ic_table(
    df: pd.DataFrame,
    indicator_columns: list,
    horizons=(60, 90),
    window_start="08:10",
    window_end="08:50",
    use_daily_ic: bool = True
) -> pd.DataFrame:
    """
    Calcula una tabla resumen de Information Coefficient (IC) para cada
    indicador técnico y cada horizonte temporal (60 / 90).

    Para cada indicador y horizonte se evalúa su relación con:
    - delta_pts_h        → magnitud del movimiento futuro (continuo)
    - trade_h            → dirección (-1, 0, +1)
    - trade_h_only       → dirección pura (excluye no-trade)
    - target_op_h        → evento operativo (0 / 1)
    - target_tail_h      → evento extremo (0 / 1)

    Parámetro clave:
    - use_daily_ic = True
        Calcula IC por día y luego promedia (más robusto estadísticamente)
    - use_daily_ic = False
        Calcula IC global usando todas las filas juntas
    """

    # 1. Filtramos el dataset únicamente a la ventana de gestación
    dfw = filter_time_window(df, window_start, window_end).copy()

    # Lista donde se acumularán los resultados fila por fila
    rows = []

    # 2. Iteramos por cada indicador técnico
    for ind in indicator_columns:

        # 3. Iteramos por cada horizonte temporal (60 y 90 minutos)
        for h in horizons:

            # Nombres de las columnas target asociadas a ese horizonte
            cols = {
                "delta": f"delta_pts_{h}",
                "trade": f"trade_{h}",
                "op": f"target_op_{h}",
                "tail": f"target_tail_{h}",
            }

            # 4. Verificamos que existan todas las columnas necesarias
            missing = [
                c for c in [ind, *cols.values()]
                if c not in dfw.columns
            ]

            # Si falta alguna columna, devolvemos NaN y seguimos
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "IC_delta": np.nan,
                    "IC_trade": np.nan,
                    "IC_trade_only": np.nan,
                    "IC_target_op": np.nan,
                    "IC_target_tail": np.nan,
                    "note": f"missing: {missing}"
                })
                continue

            # 5. Cálculo del IC (modo recomendado: diario)
            if use_daily_ic:

                # IC principal: indicador vs delta futuro
                ic_delta = daily_ic(dfw, ind, cols["delta"]).mean()

                # IC direccional: indicador vs trade (-1, 0, +1)
                ic_trade = daily_ic(dfw, ind, cols["trade"]).mean()

                # IC direccional puro: solo cuando hay trade
                df_trade_only = dfw[dfw[cols["trade"]] != 0]
                ic_trade_only = (
                    daily_ic(df_trade_only, ind, cols["trade"]).mean()
                    if len(df_trade_only) > 0 else np.nan
                )

                # IC contra evento operativo
                ic_op = daily_ic(dfw, ind, cols["op"]).mean()

                # IC contra evento extremo
                ic_tail = daily_ic(dfw, ind, cols["tail"]).mean()

            # 6. Alternativa: IC global (no recomendado, pero disponible)
            else:
                ic_delta = spearman_ic(dfw[ind], dfw[cols["delta"]])
                ic_trade = spearman_ic(dfw[ind], dfw[cols["trade"]])

                df_trade_only = dfw[dfw[cols["trade"]] != 0]
                ic_trade_only = (
                    spearman_ic(
                        df_trade_only[ind],
                        df_trade_only[cols["trade"]]
                    ) if len(df_trade_only) > 0 else np.nan
                )

                ic_op = spearman_ic(dfw[ind], dfw[cols["op"]])
                ic_tail = spearman_ic(dfw[ind], dfw[cols["tail"]])

            # 7. Guardamos los resultados para este indicador y horizonte
            rows.append({
                "indicator": ind,
                "horizon": h,
                "IC_delta": ic_delta,
                "IC_trade": ic_trade,
                "IC_trade_only": ic_trade_only,
                "IC_target_op": ic_op,
                "IC_target_tail": ic_tail,
                "note": ""
            })

    # 8. Construimos el DataFrame final
    out = pd.DataFrame(rows)

    # 9. Ranking principal: magnitud del IC contra delta
    out["abs_IC_delta"] = out["IC_delta"].abs()

    out = (
        out
        .sort_values(["horizon", "abs_IC_delta"], ascending=[True, False])
        .reset_index(drop=True)
    )

    return out

#### **3.2.2. Aplicación**

In [37]:
from pathlib import Path
import pandas as pd

# Ruta del artifact de IC
#IC_ARTIFACT_PATH = Path("reports/ic_table.parquet")
IC_ARTIFACT_PATH = DRIVE_DIR / "reports" / "ic_table.parquet"

# ------------------------------------------------------------
# Verificación previa: ¿ya existe la tabla de IC?
# ------------------------------------------------------------
if IC_ARTIFACT_PATH.exists():
    print(f"IC table encontrada. Cargando desde: {IC_ARTIFACT_PATH.resolve()}")
    ic_table = pd.read_parquet(IC_ARTIFACT_PATH)

else:
    print("IC table no encontrada. Se procederá a calcularla.")

    # ------------------------------------------------------------------
    # Cálculo de la tabla de Information Coefficient (IC)
    # ------------------------------------------------------------------
    ic_table = compute_ic_table(
        df=mnq_intraday_with_indicators,     # Dataset intradía con indicadores + targets
        indicator_columns=indicator_columns, # Lista de indicadores técnicos
        horizons=(60, 90),                    # Horizontes de predicción
        window_start="08:10",                 # Inicio ventana de gestación
        window_end="08:40",                   # Fin ventana de gestación
        use_daily_ic=True                     # IC diario promedio (recomendado)
    )

    # Guardado del artifact para no recalcular
    IC_ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    ic_table.to_parquet(IC_ARTIFACT_PATH)

    print(f"IC table calculada y guardada en: {IC_ARTIFACT_PATH.resolve()}")

# Visualización rápida
#ic_table.head(20)


IC table encontrada. Cargando desde: /content/drive/MyDrive/neural_profit/reports/ic_table.parquet


#### **3.2.3. Resultado**

In [38]:
ic_table

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta
0,price_ema60,60,-0.466803,-0.282841,-0.636263,0.015617,0.067863,,0.466803
1,price_ema30,60,-0.443975,-0.270568,-0.629694,0.013366,0.064570,,0.443975
2,bb_60_15,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
3,bb_60_20,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
4,bb_60_25,60,-0.425702,-0.252731,-0.569178,0.020030,0.073461,,0.425702
...,...,...,...,...,...,...,...,...,...
79,volume_ratio_15,90,0.013434,0.024266,0.069968,-0.003612,-0.007500,,0.013434
80,volume_ratio_30,90,0.013173,0.025973,0.101723,-0.024943,-0.031617,,0.013173
81,volume_ratio_20,90,0.011937,0.025551,0.096460,-0.011576,-0.017551,,0.011937
82,volume_ratio_90,90,0.010410,0.020027,0.086906,-0.023809,-0.044796,,0.010410


### **3.3. Separación por horizonte**

In [39]:
ic_60 = ic_table[ic_table["horizon"] == 60].copy()
ic_90 = ic_table[ic_table["horizon"] == 90].copy()

### **3.4. Primer filtro: fuerza miníma de señal**

Para el análisis intradía se adopta el siguiente criterio empírico de interpretación del Information Coefficient (IC):

- |IC| < 0.02 → ruido
- 0.02 ≤ |IC| < 0.05 → débil
- 0.05 ≤ |IC| < 0.10 → moderado
- |IC| ≥ 0.10 → fuerte

Dado que el objetivo de este stage es identificar señales con capacidad predictiva real, se descartan aquellas cuyo |IC| se encuentra por debajo del umbral de relevancia. En consecuencia, se conservan únicamente los indicadores que presentan una señal fuerte, definida como:

$$|IC_Δ|≥0.10$$

Este primer filtro elimina indicadores dominados por ruido y reduce el espacio de features a un conjunto con relación estadísticamente significativa respecto a la magnitud del movimiento futuro.

In [40]:
# Filtro por fuerza mínima de señal (|IC_delta| >= 0.10)
ic_60_relevant = ic_60[ic_60["abs_IC_delta"] >= 0.10].copy()
ic_90_relevant = ic_90[ic_90["abs_IC_delta"] >= 0.10].copy()

In [41]:
# Extraer lista única de indicadores técnicos relevantes
ti_ic_60_relevant_list = (
    ic_60["indicator"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

ti_ic_90_relevant_list = (
    ic_90["indicator"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)


### **3.5. Conclusión preliminar**

Se realizó el análisis de Information Coefficient para la totalidad de los indicadores técnicos calculados, evaluando su relación con el target de retorno futuro. Como criterio de relevancia, se seleccionaron únicamente aquellos indicadores con IC > 0.1.

El resultado muestra que, tanto para H = 60 minutos como para H = 90 minutos, el conjunto de indicadores relevantes es exactamente el mismo, lo que evidencia una consistencia estructural de las señales técnicas independientemente del horizonte analizado. Las listas ti_ic_60_relevant_list y ti_ic_90_relevant_list convergen en un único conjunto de indicadores con poder predictivo estable.

In [42]:
print('TI H=60: ', ti_ic_60_relevant_list)
print('TI H=90: ', ti_ic_90_relevant_list)


TI H=60:  ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'macd', 'momentum_10', 'momentum_3', 'momentum_5', 'price_ema15', 'price_ema20', 'price_ema30', 'price_ema60', 'roc_10', 'roc_20', 'roc_30', 'roc_5', 'roc_60', 'rsi_14', 'rsi_3', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']
TI H=90:  ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'macd', 'momentum_10', 'momentum_3', 'momentum_5', 'price_ema15', 'price_ema20', 'price_ema30', 'price_ema60', 'roc_10', 'roc_20', 'roc_30', 'roc_5', 'roc_60', 'rsi_14', 'rsi_3', 'rsi_5', 'rsi_7', 'stoch_k_14

Dado que el set relevante es común a ambos horizontes, el siguiente paso consiste en analizar la correlación entre los indicadores seleccionados y con el target, con el objetivo de identificar redundancias, colinealidad y seleccionar un subconjunto final informativamente eficiente.

## **4. Correlación**

### **4.1. Correlación de indicadores técnicos**

Una vez identificado el conjunto de indicadores técnicos relevantes mediante el análisis de Information Coefficient, se procede a estudiar la estructura de dependencia entre ellos.
El objetivo de este análisis es cuantificar el grado de colinealidad y redundancia informativa dentro del set seleccionado, así como comprender cómo se relacionan las distintas familias de indicadores entre sí.

Para ello, se calcula la matriz de correlación Spearman promedio por día, restringida a una ventana horaria específica, la ventana de gestación (08:10 - 08:40), lo que permite capturar relaciones monótonas y robustas a outliers, preservando la consistencia intradía del comportamiento del mercado.

Este análisis constituye un paso clave previo a la selección final de features y a su utilización conjunta en modelos predictivos.

#### **4.1.1. Implementación para calculo de Correlación**

In [43]:
import numpy as np
import pandas as pd

def daily_spearman_corr_matrix(
    df: pd.DataFrame,
    indicator_columns: list[str],
    window_start: str = "08:10",
    window_end: str = "08:40",
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Calcula la matriz de correlación Spearman PROMEDIO por día,
    restringida a la ventana horaria [window_start, window_end].

    Requisitos:
      - df index: DatetimeIndex
      - df contiene columna date_col
      - indicator_columns existen en df

    Devuelve:
      - corr_mean: DataFrame NxN con promedio de Spearman por día
    """
    # 1) Filtrar ventana horaria (por índice temporal)
    df_win = df.between_time(window_start, window_end).copy()

    # 2) Validaciones mínimas
    missing = [c for c in indicator_columns if c not in df_win.columns]
    if missing:
        raise ValueError(f"Faltan columnas en df: {missing}")

    # 3) Correlación Spearman por día (matriz)
    daily_corr = []
    for d, g in df_win.groupby(date_col):
        X = g[indicator_columns]

        # Si el día tiene demasiados NaN, Spearman puede devolver NaN
        corr = X.corr(method="spearman")

        # Guardar solo si tiene algo útil
        if corr.notna().values.any():
            daily_corr.append(corr)

    if not daily_corr:
        raise ValueError("No se pudieron calcular correlaciones (revisar ventana, NaNs o columnas).")

    # 4) Promedio de matrices (alineadas por índice/columnas)
    corr_mean = sum(daily_corr) / len(daily_corr)

    return corr_mean


def top_abs_correlations(
    corr: pd.DataFrame,
    top_n: int = 15,
    min_abs_rho: float = 0.0,
) -> pd.DataFrame:
    """
    Devuelve el ranking de pares con mayor |rho| (sin duplicados i-j y sin diagonal).
    """
    c = corr.copy()

    # Enmascarar diagonal y duplicados (triángulo inferior)
    mask = np.tril(np.ones(c.shape, dtype=bool))
    c = c.mask(mask)

    # Pasar a formato largo
    pairs = (
        c.stack()
         .rename("rho")
         .reset_index()
         .rename(columns={"level_0": "indicator_1", "level_1": "indicator_2"})
    )

    # |rho| y filtro
    pairs["abs_rho"] = pairs["rho"].abs()
    pairs = pairs[pairs["abs_rho"] >= min_abs_rho]

    return pairs.sort_values("abs_rho", ascending=False).head(top_n).reset_index(drop=True)


#### **4.1.2. Aplicación de cálculo de Correlación**

In [44]:
corr_mean_ic_60_relevant = daily_spearman_corr_matrix(
    df=mnq_intraday_with_indicators,
    indicator_columns=ti_ic_60_relevant_list,
    window_start="08:10",
    window_end="08:40",
)

In [45]:
corr_mean_ic_90_relevant = daily_spearman_corr_matrix(
    df=mnq_intraday_with_indicators,
    indicator_columns=ti_ic_90_relevant_list,
    window_start="08:10",
    window_end="08:40",
)

In [46]:
#Quitar # si desea visualizar la matriz completa
#corr_mean_ic_60_relevant
corr_mean_ic_60_relevant.head()

,atr_norm_10,atr_norm_14,atr_norm_20,atr_norm_30,atr_norm_5,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,...,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30,volume_ratio_15,volume_ratio_20,volume_ratio_30,volume_ratio_60,volume_ratio_90
atr_norm_10,1.000000,0.980442,0.939385,0.880834,0.936832,-0.012796,-0.012796,-0.012796,-0.021681,-0.021681,...,-0.014338,-0.024697,-0.022608,-0.039045,-0.053797,0.296326,0.357421,0.420467,0.479629,0.499435
atr_norm_14,0.980442,1.000000,0.978381,0.929451,0.886620,-0.005756,-0.005756,-0.005756,-0.013164,-0.013164,...,-0.006732,-0.015941,-0.014294,-0.030294,-0.045277,0.248954,0.307093,0.369851,0.435122,0.457457
atr_norm_20,0.939385,0.978381,1.000000,0.973552,0.830625,0.001050,0.001050,0.001050,-0.004339,-0.004339,...,-0.000120,-0.007603,-0.006555,-0.021377,-0.035918,0.215804,0.268624,0.327251,0.395112,0.419896
atr_norm_30,0.880834,0.929451,0.973552,1.000000,0.771005,0.005349,0.005349,0.005349,0.001834,0.001834,...,0.003322,-0.002481,-0.001605,-0.015126,-0.028351,0.193824,0.240652,0.292259,0.357891,0.384113
atr_norm_5,0.936832,0.886620,0.830625,0.771005,1.000000,-0.031802,-0.031802,-0.031802,-0.040593,-0.040593,...,-0.033914,-0.043951,-0.042546,-0.055495,-0.066215,0.429398,0.486404,0.537665,0.581773,0.596731


In [47]:
#Quitar # si desea visualizar la matriz completa
#corr_mean_ic_90_relevant
corr_mean_ic_90_relevant.head()

,atr_norm_10,atr_norm_14,atr_norm_20,atr_norm_30,atr_norm_5,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,...,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30,volume_ratio_15,volume_ratio_20,volume_ratio_30,volume_ratio_60,volume_ratio_90
atr_norm_10,1.000000,0.980442,0.939385,0.880834,0.936832,-0.012796,-0.012796,-0.012796,-0.021681,-0.021681,...,-0.014338,-0.024697,-0.022608,-0.039045,-0.053797,0.296326,0.357421,0.420467,0.479629,0.499435
atr_norm_14,0.980442,1.000000,0.978381,0.929451,0.886620,-0.005756,-0.005756,-0.005756,-0.013164,-0.013164,...,-0.006732,-0.015941,-0.014294,-0.030294,-0.045277,0.248954,0.307093,0.369851,0.435122,0.457457
atr_norm_20,0.939385,0.978381,1.000000,0.973552,0.830625,0.001050,0.001050,0.001050,-0.004339,-0.004339,...,-0.000120,-0.007603,-0.006555,-0.021377,-0.035918,0.215804,0.268624,0.327251,0.395112,0.419896
atr_norm_30,0.880834,0.929451,0.973552,1.000000,0.771005,0.005349,0.005349,0.005349,0.001834,0.001834,...,0.003322,-0.002481,-0.001605,-0.015126,-0.028351,0.193824,0.240652,0.292259,0.357891,0.384113
atr_norm_5,0.936832,0.886620,0.830625,0.771005,1.000000,-0.031802,-0.031802,-0.031802,-0.040593,-0.040593,...,-0.033914,-0.043951,-0.042546,-0.055495,-0.066215,0.429398,0.486404,0.537665,0.581773,0.596731


#### **4.1.3. Observaciones clave (correlación Spearman promedio)**

- Estructura prácticamente idéntica en 60 y 90 min: la matriz es la misma en magnitudes y patrones, lo que confirma que las dependencias entre indicadores no dependen del horizonte, al menos en la ventana analizada.

- Bloque ATR muy colineal: `atr_norm_{5,10,14,20,30}` muestra correlaciones muy altas entre sí (≈0.77 a 0.98). Señal clara de redundancia.

- Bloque Bollinger casi duplicado: combinaciones `bb_*_*` aparecen con correlaciones ≈0.78–1.00, incluso con valores 1.00 entre variantes (misma información con diferente parametrización).

- Indicadores de tendencia/retorno altamente solapados: `momentum_*` y `roc_*` presentan correlaciones elevadas con el bloque BB/EMA/MACD (típicamente ~0.5–0.8), indicando que capturan una dinámica muy similar (movimiento/tendencia).

- Osciladores también redundantes: `rsi_{3,5,7,14}` y `stoch_k_{14,20,30}` están fuertemente correlacionados entre sí y con BB/EMA (muchos valores >0.85).

- Volumen como bloque independiente pero colineal internamente: `volume_ratio_{15,20,30,60,90}` tiene correlaciones muy altas entre sí (≈0.91–0.99) y correlación moderada con ATR (≈0.25–0.60), sugiriendo relación con “régimen de actividad/volatilidad”.

- Separación por familias: se observa una estructura típica de clusters naturales:

    (ATR/volatilidad) ↔ (BB/EMA/MACD/tendencia) ↔ (RSI/Stoch/osc.) ↔ (Volumen).

###**4.2. Clusters de Correlación**


Dado el alto nivel de colinealidad observado entre numerosos indicadores (y la redundancia entre distintas parametrizaciones), se construyen clusters de indicadores para agrupar señales equivalentes y obtener un set final más eficiente. Este procedimiento permite:

- agrupar indicadores que capturan esencialmente la misma información,
- reducir dimensionalidad sin perder señal,
- seleccionar representantes por cluster (o combinar señales) para disminuir el riesgo de sobreajuste,
- mejorar la interpretabilidad y robustez del conjunto de features.

Definición conceptual (en términos de grafo):

- Cada indicador es un nodo.

- Existe una arista entre dos indicadores i,j si su correlación absoluta cumple:   $|𝜌_{ij} \ge 0.85|$

- Cada componente conexa del grafo es un cluster informativo.

En otras palabras, cada cluster representa un grupo de indicadores que comparten prácticamente la misma información y, por lo tanto, pueden tratarse como una misma “familia” operativa a efectos de selección de features.

#### **4.2.1. Código para construcción de clusters**

Este código:
- toma la matriz de correlación media (corr_mean_ic_h_relevant)
- detecta clusters automáticamente
- devuelve un DataFrame limpio para inspección

In [48]:
import numpy as np
import pandas as pd
import networkx as nx


def build_correlation_clusters(
    corr: pd.DataFrame,
    threshold: float = 0.85,
) -> pd.DataFrame:
    """
    Construye clusters de indicadores basados en alta correlación (|rho| >= threshold).

    Parameters
    ----------
    corr : pd.DataFrame
        Matriz de correlación (Spearman promedio), índice = columnas = indicadores.
    threshold : float
        Umbral de |rho| para considerar redundancia.

    Returns
    -------
    clusters_df : pd.DataFrame
        Tabla con columnas:
        - cluster_id
        - indicator
        - n_in_cluster
    """

    # 1) Crear grafo
    G = nx.Graph()

    indicators = corr.columns.tolist()
    G.add_nodes_from(indicators)

    # 2) Agregar aristas si |rho| >= threshold (excluye diagonal)
    for i in indicators:
        for j in indicators:
            if i == j:
                continue
            rho = corr.loc[i, j]
            if pd.notna(rho) and abs(rho) >= threshold:
                G.add_edge(i, j, weight=rho)

    # 3) Extraer componentes conexas (clusters)
    components = list(nx.connected_components(G))

    # 4) Armar tabla resultado
    records = []
    for cluster_id, comp in enumerate(components):
        comp = sorted(list(comp))
        for ind in comp:
            records.append({
                "cluster_id": cluster_id,
                "indicator": ind,
                "n_in_cluster": len(comp),
            })

    clusters_df = pd.DataFrame(records).sort_values(
        ["n_in_cluster", "cluster_id"],
        ascending=[False, True]
    ).reset_index(drop=True)

    return clusters_df


#### **4.2.2. Aplicación**

- `cluster_id`: identifica cada grupo informativo
- `n_in_cluster`:
  - 1 → indicador independiente
  - $>$ 1 → redundancia fuerte

In [49]:
clusters_ic_60 = build_correlation_clusters(
    corr=corr_mean_ic_60_relevant,
    threshold=0.85
)

clusters_ic_90 = build_correlation_clusters(
    corr=corr_mean_ic_90_relevant,
    threshold=0.85
)


In [50]:
clusters_ic_60.head()
clusters_ic_90.head()

,cluster_id,indicator,n_in_cluster
0,1,bb_15_15,23
1,1,bb_15_20,23
2,1,bb_15_25,23
3,1,bb_20_15,23
4,1,bb_20_20,23


#### **4.2.3. Observaciones de clusters**

El análisis de correlación permitió identificar bloques informativos bien definidos entre los indicadores técnicos que superaron el umbral de relevancia por IC.

- Cluster dominante (n = 23)
Incluye indicadores de tendencia y osciladores derivados del precio (EMA, Bollinger Bands, RSI y Stochastic).
Todos ellos están altamente correlacionados y capturan esencialmente la misma información económica: el estado o extensión del precio respecto a su dinámica reciente.
No representan señales independientes, sino reparametrizaciones de un mismo régimen, por lo que este cluster no puede incorporarse completo al modelo. Será necesario seleccionar un único representante.

- Cluster de volatilidad (n = 5)
Formado por atr_norm_*. Mide volatilidad pura.
Según el análisis de IC, su relación directa con la magnitud del movimiento es débil. Se identifica como un bloque informativo separado, potencialmente útil como contexto, pero no como señal principal en esta etapa.

- Cluster de volumen (n = 5)
Incluye volume_ratio_*. Aporta información distinta al precio, pero con IC bajo.
Se mantiene como bloque independiente, a evaluar más adelante, sin integrarlo aún al núcleo del modelo.

- Cluster de momentum medio (n = 3)
Compuesto por macd, momentum_10 y roc_10.
Representa señales de momentum y cambio de régimen, conceptualmente distintas del estado del precio. Es un cluster informativo relevante que, por redundancia interna, suele resolverse quedándose con un solo indicador.

- Clusters pequeños e indicadores individuales
Incluyen momentum en distintas escalas temporales (momentum_*, roc_*) que no presentan alta correlación con el cluster dominante.
Estos indicadores aportan información complementaria y capturan dinámicas de momentum específicas por horizonte.

###**4.3. Procesamiento de clusters**


#### **4.3.1. Resolución de cluster dominante**

Una vez identificados los clusters por alta correlación (redundancia), el objetivo es conservar un solo indicador por cluster, eligiendo el que tenga mayor potencia predictiva.

Para eso, dentro de cada `cluster_id` se selecciona el indicador con mayor `abs_IC_delta` (y, en caso de empate, mayor `abs(IC_trade_only)`).

#### **4.3.2. Código**

In [51]:
import numpy as np
import pandas as pd

def resolve_clusters_by_ic(
    ic_relevant: pd.DataFrame,
    clusters_df: pd.DataFrame,
    corr: pd.DataFrame | None = None,
    prefer_trade_only_tiebreak: bool = True
) -> pd.DataFrame:
    """
    Resuelve clusters de redundancia seleccionando 1 indicador por cluster,
    usando dominancia de IC.

    Selección:
      1) Mayor abs_IC_delta
      2) (opcional) Si hay empate: mayor abs(IC_trade_only)

    Devuelve una tabla con:
      - cluster_id, n_in_cluster, indicator, keep, reason, métricas IC
    """
    # Asegurar columnas necesarias
    needed_ic = {"indicator", "abs_IC_delta", "IC_trade_only", "IC_delta"}
    needed_cl = {"cluster_id", "indicator", "n_in_cluster"}
    if not needed_ic.issubset(ic_relevant.columns):
        raise ValueError(f"ic_relevant debe contener: {sorted(needed_ic)}")
    if not needed_cl.issubset(clusters_df.columns):
        raise ValueError(f"clusters_df debe contener: {sorted(needed_cl)}")

    # Merge IC + cluster info
    df = clusters_df.merge(
        ic_relevant[["indicator", "IC_delta", "IC_trade_only", "abs_IC_delta"]],
        on="indicator",
        how="left"
    )

    # Validar que todos los indicadores tengan IC
    missing_ic = df[df["abs_IC_delta"].isna()]["indicator"].unique().tolist()
    if missing_ic:
        raise ValueError(f"Estos indicadores están en clusters pero no en ic_relevant: {missing_ic}")

    # Resolver por cluster
    out_rows = []
    for cid, g in df.groupby("cluster_id", sort=True):
        g = g.copy()

        # criterio principal
        max_abs = g["abs_IC_delta"].max()
        candidates = g[g["abs_IC_delta"] == max_abs].copy()

        # desempate opcional
        if prefer_trade_only_tiebreak and len(candidates) > 1:
            candidates["abs_IC_trade_only"] = candidates["IC_trade_only"].abs()
            best = candidates.sort_values("abs_IC_trade_only", ascending=False).iloc[0]
        else:
            best = candidates.iloc[0]

        best_indicator = best["indicator"]

        # armar salida
        for _, row in g.iterrows():
            keep = row["indicator"] == best_indicator
            reason = ""
            if keep:
                reason = "max abs_IC_delta"
                if prefer_trade_only_tiebreak and len(candidates) > 1:
                    reason += " (tiebreak: abs_IC_trade_only)"
            else:
                reason = f"redundant (same cluster as {best_indicator})"

            out_rows.append({
                "cluster_id": int(cid),
                "n_in_cluster": int(row["n_in_cluster"]),
                "indicator": row["indicator"],
                "keep": bool(keep),
                "reason": reason,
                "IC_delta": float(row["IC_delta"]),
                "IC_trade_only": float(row["IC_trade_only"]),
                "abs_IC_delta": float(row["abs_IC_delta"]),
            })

    resolved = pd.DataFrame(out_rows).sort_values(
        ["cluster_id", "keep", "abs_IC_delta"],
        ascending=[True, False, False]
    ).reset_index(drop=True)

    return resolved


#### **4.3.3. Aplicación**

In [52]:
# 1) Filtrar clusters a los indicadores relevantes (por si clusters_df incluye otros)
clusters_ic_60_relevant = clusters_ic_60[
    clusters_ic_60["indicator"].isin(ic_60_relevant["indicator"])
].copy()

# 2) Resolver clusters
resolved_60 = resolve_clusters_by_ic(
    ic_relevant=ic_60_relevant,
    clusters_df=clusters_ic_60_relevant,
    prefer_trade_only_tiebreak=True
)

# 3) Lista final de indicadores
final_indicators_60 = resolved_60.loc[resolved_60["keep"], "indicator"].tolist()

# 4) Ver qué se conserva
resolved_60[resolved_60["keep"]].sort_values("abs_IC_delta", ascending=False)

,cluster_id,n_in_cluster,indicator,keep,reason,IC_delta,IC_trade_only,abs_IC_delta
0,1,23,price_ema60,True,max abs_IC_delta,-0.466803,-0.636263,0.466803
31,7,1,roc_60,True,max abs_IC_delta,-0.392433,-0.581905,0.392433
30,6,1,roc_30,True,max abs_IC_delta,-0.386184,-0.552156,0.386184
29,5,1,roc_20,True,max abs_IC_delta,-0.377138,-0.577925,0.377138
23,2,3,momentum_10,True,max abs_IC_delta (tiebreak: abs_IC_trade_only),-0.361038,-0.550246,0.361038
27,4,2,momentum_5,True,max abs_IC_delta (tiebreak: abs_IC_trade_only),-0.293909,-0.436008,0.293909
26,3,1,momentum_3,True,max abs_IC_delta,-0.241331,-0.299525,0.241331


In [53]:
# 1) Filtrar clusters a los indicadores relevantes (por si clusters_df incluye otros)
clusters_ic_90_relevant = clusters_ic_90[
    clusters_ic_90["indicator"].isin(ic_90_relevant["indicator"])
].copy()

# 2) Resolver clusters
resolved_90 = resolve_clusters_by_ic(
    ic_relevant=ic_90_relevant,
    clusters_df=clusters_ic_90_relevant,
    prefer_trade_only_tiebreak=True
)

# 3) Lista final de indicadores
final_indicators_90 = resolved_90.loc[resolved_90["keep"], "indicator"].tolist()

# 4) Ver qué se conserva
resolved_90[resolved_90["keep"]].sort_values("abs_IC_delta", ascending=False)

,cluster_id,n_in_cluster,indicator,keep,reason,IC_delta,IC_trade_only,abs_IC_delta
0,1,23,price_ema60,True,max abs_IC_delta,-0.304082,-0.583596,0.304082
30,6,1,roc_30,True,max abs_IC_delta,-0.255689,-0.513596,0.255689
31,7,1,roc_60,True,max abs_IC_delta,-0.250912,-0.515727,0.250912
29,5,1,roc_20,True,max abs_IC_delta,-0.250544,-0.534399,0.250544
23,2,3,momentum_10,True,max abs_IC_delta (tiebreak: abs_IC_trade_only),-0.246508,-0.531273,0.246508
27,4,2,momentum_5,True,max abs_IC_delta (tiebreak: abs_IC_trade_only),-0.209218,-0.463464,0.209218
26,3,1,momentum_3,True,max abs_IC_delta,-0.175457,-0.345358,0.175457


#### **4.3.4. Comentarios generales – Resolución de clusters**

In [54]:
print('TI finales H=60:', final_indicators_60)
print('TI finales H=90:', final_indicators_90)

TI finales H=60: ['price_ema60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']
TI finales H=90: ['price_ema60', 'momentum_10', 'momentum_3', 'momentum_5', 'roc_20', 'roc_30', 'roc_60']


- La construcción de clusters permite reducir el conjunto original a 7 indicadores representativos, uno por cluster, eliminando redundancia informativa y alta colinealidad.

- `price_ema60` se consolida como el indicador dominante del bloque de tendencia, presentando el mayor $|IC_Δ|$ de forma consistente.
- Los indicadores ROC (20, 30 y 60) representan de manera robusta el bloque de retornos normalizados, con señales estables y coherentes.
- El bloque de momentum (3, 5 y 10) captura la aceleración del precio, siendo `momentum_10` el representante más informativo dentro de su cluster.
- El criterio de selección basado en el máximo $|IC_Δ|$ (con desempate por `IC_trade_only`) garantiza alineación directa con el objetivo predictivo y estabilidad del set final.
- La coincidencia estructural de los clusters y de los indicadores seleccionados refuerza la robustez y generalidad del conjunto de features resultante.

### **4.4. Correlación final**

Una vez reducido el conjunto de indicadores técnicos mediante el filtrado por fuerza de señal (|IC| ≥ 0.10) y la resolución por clusters, analizaremos la matriz de correlación entre los indicadores resultantes con el objetivo de evaluar posibles redundancias adicionales.

In [55]:
corr_final = daily_spearman_corr_matrix(
    df=mnq_intraday_with_indicators,
    indicator_columns=final_indicators_90,
    window_start="08:10",
    window_end="08:40",
)

corr_final

,price_ema60,momentum_10,momentum_3,momentum_5,roc_20,roc_30,roc_60
price_ema60,1.000000,0.770346,0.537972,0.647685,0.780107,0.772257,0.705180
momentum_10,0.770346,1.000000,0.466833,0.612312,0.593234,0.535808,0.511864
momentum_3,0.537972,0.466833,1.000000,0.693103,0.357253,0.348236,0.341067
momentum_5,0.647685,0.612312,0.693103,1.000000,0.447330,0.427472,0.418073
roc_20,0.780107,0.593234,0.357253,0.447330,1.000000,0.617520,0.541595
roc_30,0.772257,0.535808,0.348236,0.427472,0.617520,1.000000,0.571490
roc_60,0.705180,0.511864,0.341067,0.418073,0.541595,0.571490,1.000000


En primer lugar, se observa que no existen correlaciones extremas (|ρ| ≥ 0.85) entre los indicadores seleccionados. Esto indica ausencia de colinealidad severa y confirma que no es necesario eliminar variables de forma automática por problemas de multicolinealidad.

No obstante, sí aparecen correlaciones moderadas a altas (≈ 0.70–0.78) en algunos pares específicos. En particular:

- `price_ema60` presenta correlaciones elevadas con distintos indicadores de tipo ROC (`roc_20`, `roc_30`, `roc_60`).

- Dentro de la familia de momentum, se observa una correlación relativamente alta entre `momentum_3` y `momentum_5`.

Este comportamiento refleja un solapamiento parcial de información, esperable dado que estos indicadores describen aspectos relacionados de la dinámica del precio, pero no implica duplicación completa de señal.

- Desde una perspectiva conceptual, considero que `price_ema60` cumple un rol estructural, ya que resume el estado tendencial y la extensión relativa del precio. A pesar de su correlación con otros indicadores, presenta el mayor Information Coefficient del conjunto, por lo que resulta justificable mantenerlo como feature principal.

- Asimismo, los indicadores de momentum y rate of change (ROC) no son equivalentes: el primero captura cambios absolutos, mientras que el segundo mide variaciones relativas. Las correlaciones cruzadas entre ambos grupos son moderadas, lo que justifica su coexistencia siempre que se evite redundancia de escala.

Finalmente, el análisis confirma que la principal fuente de redundancia se encuentra en las múltiples escalas temporales dentro de una misma familia, más que en el concepto subyacente del indicador.

### **4.4. Decisión de consolidación final**

En base a este análisis, decido realizar una reducción adicional por escala, manteniendo una única variante representativa por familia, priorizando:

- estabilidad estadística,
- interpretabilidad económica,
- y alineación con el horizonte de predicción.

Este criterio permite obtener un set de indicadores compacto, informativamente eficiente y metodológicamente defendible, adecuado para las etapas posteriores del pipeline.

In [56]:
technical_indicators_finals = ["price_ema60", "momentum_10",  "roc_30", "roc_60"]

## **5. Análisis de coherencia direccional**

Una vez consolidado el conjunto final de indicadores técnicos:

```
technical_indicators_finals = [
    "price_ema60",
    "momentum_10",
    "roc_30",
    "roc_60",
]
```
el siguiente paso consiste en evaluar la coherencia direccional de cada indicador, contrastando dos dimensiones complementarias:

- IC_delta: relación del indicador con la magnitud del movimiento futuro.

- IC_trade_only: relación del indicador con la dirección del movimiento en instantes donde existe trade.

Este análisis permite responder una pregunta clave:

    ¿El indicador señala la misma dirección cuando explica magnitud del movimiento y cuando explica dirección operativa?

**Criterio de coherencia**

Un indicador se considera coherente si:

- `sign(IC_delta)` == `sign(IC_trade_only)`
- ninguno de los dos es 0 o NaN

Adicionalmente, se analiza la relación de intensidades:

- si `|IC_trade_only|` > `|IC_delta|`, el indicador refuerza su señal en contextos de decisión operativa, indicando mayor utilidad práctica.

Este criterio permite validar no solo la significancia estadística, sino también la consistencia económica y operativa de las señales seleccionadas.

### **5.1. Código para análisis de coherencia direccional**

In [57]:
import numpy as np
import pandas as pd

def check_directional_coherence(df_selected: pd.DataFrame) -> pd.DataFrame:
    """
    Verifica coherencia direccional entre IC_delta e IC_trade_only.

    Criterio:
      - coherent = True si sign(IC_delta) == sign(IC_trade_only) y ninguno es 0/NaN
      - coherent = False en caso contrario

    Nota:
      - La columna 'family' es opcional. Si no existe, se omite del output.
    """
    out = df_selected.copy()

    # Signos (NaN -> NaN)
    out["sign_IC_delta"] = np.sign(out["IC_delta"])
    out["sign_IC_trade_only"] = np.sign(out["IC_trade_only"])

    # Coherencia: mismos signos y ambos distintos de 0 y no NaN
    out["coherent"] = (
        out["IC_delta"].notna()
        & out["IC_trade_only"].notna()
        & (out["sign_IC_delta"] != 0)
        & (out["sign_IC_trade_only"] != 0)
        & (out["sign_IC_delta"] == out["sign_IC_trade_only"])
    )

    # Comparación de magnitudes
    out["abs_IC_trade_only"] = out["IC_trade_only"].abs()
    out["abs_IC_delta"] = out["IC_delta"].abs()  # por si no estuviera
    out["abs_gap_trade_vs_delta"] = out["abs_IC_trade_only"] - out["abs_IC_delta"]

    # Orden
    out = out.sort_values(["coherent", "abs_IC_delta"], ascending=[True, False])

    # Columnas base a devolver
    cols = [
        "indicator", "IC_delta", "IC_trade_only",
        "sign_IC_delta", "sign_IC_trade_only", "coherent",
        "abs_IC_delta", "abs_IC_trade_only", "abs_gap_trade_vs_delta"
    ]

    # Si existe 'family', la incluimos al inicio
    if "family" in out.columns:
        cols = ["family"] + cols

    return out[cols]

### **5.2. Aplicación**

In [58]:
ic_selected_60 = ic_table[
    (ic_table["horizon"] == 60) &
    (ic_table["indicator"].isin(technical_indicators_finals))
].copy()

coherence_60 = check_directional_coherence(ic_selected_60)

In [59]:
ic_selected_90 = ic_table[
    (ic_table["horizon"] == 90) &
    (ic_table["indicator"].isin(technical_indicators_finals))
].copy()

coherence_90 = check_directional_coherence(ic_selected_90)


### **5.3. Resultados y análisis**

In [60]:
coherence_60

,indicator,IC_delta,IC_trade_only,sign_IC_delta,sign_IC_trade_only,coherent,abs_IC_delta,abs_IC_trade_only,abs_gap_trade_vs_delta
0,price_ema60,-0.466803,-0.636263,-1.0,-1.0,True,0.466803,0.636263,0.169460
11,roc_60,-0.392433,-0.581905,-1.0,-1.0,True,0.392433,0.581905,0.189472
12,roc_30,-0.386184,-0.552156,-1.0,-1.0,True,0.386184,0.552156,0.165972
19,momentum_10,-0.361038,-0.550246,-1.0,-1.0,True,0.361038,0.550246,0.189207


In [61]:
coherence_90

,indicator,IC_delta,IC_trade_only,sign_IC_delta,sign_IC_trade_only,coherent,abs_IC_delta,abs_IC_trade_only,abs_gap_trade_vs_delta
42,price_ema60,-0.304082,-0.583596,-1.0,-1.0,True,0.304082,0.583596,0.279515
53,roc_30,-0.255689,-0.513596,-1.0,-1.0,True,0.255689,0.513596,0.257907
55,roc_60,-0.250912,-0.515727,-1.0,-1.0,True,0.250912,0.515727,0.264815
60,momentum_10,-0.246508,-0.531273,-1.0,-1.0,True,0.246508,0.531273,0.284766


El análisis de coherencia direccional se aplicó sobre el conjunto final de indicadores técnicos seleccionados (`price_ema60`, `momentum_10`, `roc_30`, `roc_60`) para los horizontes de 60 y 90 minutos, comparando la señal asociada a la magnitud del movimiento (`IC_delta`) y a la dirección operativa cuando hay trade (`IC_trade_only`).

Resultados principales

- Coherencia perfecta en ambos horizontes

  En todos los casos se cumple que `sign(IC_delta) = sign(IC_trade_only)` y ambos son distintos de cero.
  Esto indica que los indicadores no mezclan regímenes y mantienen una interpretación direccional consistente entre magnitud y decisión operativa.

- Régimen dominante de reversión

  Todos los indicadores presentan valores negativos tanto en `IC_delta` como en `IC_trade_only`.
  Esto confirma que, durante la ventana de gestación, las extensiones del precio tienden a anticipar agotamiento o reversión en los horizontes analizados (60 y 90 minutos).

- Señal más fuerte en contexto operativo

  En todos los indicadores se observa que $$∣𝐼𝐶_{𝑡𝑟𝑎𝑑𝑒-only}| >  ∣𝐼𝐶_{\Delta}| $$
  
  lo que implica que la señal es más intensa cuando existe una oportunidad de trade, reforzando su utilidad práctica.

- Estabilidad inter-horizonte

  El patrón observado en 60 minutos se replica de forma consistente en 90 minutos, tanto en signo como en magnitudes relativas.
  Esto sugiere que el set de indicadores captura una estructura temporal robusta, no dependiente de un único horizonte.




### **5.4. Conclusión operativa**



  El conjunto de indicadores técnicos seleccionado presenta coherencia direccional total, estabilidad entre horizontes y una señal más fuerte en contextos operativos reales.
  
  Esto valida su uso como núcleo definitivo de features para el modelo, sin ambigüedad de régimen y con interpretación económica clara.

In [62]:
technical_indicators_finals

['price_ema60', 'momentum_10', 'roc_30', 'roc_60']

## **6. Análisis de OHLCV**

Las variables OHLCV (`open`, `high`, `low`, `close`, `volume`) representan el estado instantáneo del mercado y constituyen la materia prima a partir de la cual se construyen los indicadores técnicos.

Aunque no están diseñadas explícitamente como señales anticipatorias, su inclusión directa en el modelo puede:

- aportar información contextual relevante,
- introducir redundancia respecto a los indicadores derivados,
- o, en algunos casos, mostrar capacidad predictiva directa.

Por este motivo, se realiza un análisis específico de Information Coefficient (IC) y correlación para las variables OHLCV, con el objetivo de evaluar su contribución real y justificar su inclusión en el modelo.

### **6.1. IC de variables OHLCV vs targets**


Se evalúa la relación entre OHLCV y los targets definidos (`delta_pts_h`, `trade_h`, etc.) en la ventana de gestación, utilizando el mismo criterio metodológico aplicado a los indicadores técnicos.


In [107]:
mnq_intraday_with_indicators.head()

,date,open,high,low,close,volume,delta_pts_60,trade_60,target_op_60,target_tail_60,...,atr_norm_5,atr_norm_10,atr_norm_14,atr_norm_20,atr_norm_30,roc_5,roc_10,roc_20,roc_30,roc_60
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-23 07:59:00-05:00,2019-12-23,8734.00,8734.25,8734.00,8734.00,47,4.25,0,0,0,...,0.000093,0.000095,0.000097,0.000101,0.000106,-0.008586,0.011451,0.000000,-0.022894,0.077344
2019-12-23 08:00:00-05:00,2019-12-23,8734.00,8734.25,8733.75,8733.75,31,3.50,0,0,0,...,0.000086,0.000091,0.000094,0.000099,0.000105,-0.017172,0.011451,0.000000,-0.034338,0.065880
2019-12-23 08:01:00-05:00,2019-12-23,8734.00,8734.25,8733.75,8734.00,16,4.00,0,0,0,...,0.000080,0.000087,0.000091,0.000097,0.000103,-0.011448,0.011451,0.005725,-0.020033,0.080211
2019-12-23 08:02:00-05:00,2019-12-23,8734.00,8734.00,8733.25,8733.25,23,4.75,0,0,0,...,0.000081,0.000087,0.000091,0.000096,0.000103,-0.020034,-0.002863,-0.002863,-0.020034,0.060151
2019-12-23 08:03:00-05:00,2019-12-23,8734.25,8734.50,8734.00,8734.00,23,3.75,0,0,0,...,0.000094,0.000093,0.000095,0.000099,0.000104,0.000000,-0.017171,0.014314,-0.005724,0.065878


In [ ]:
"""
    Calcula una tabla resumen de Information Coefficient (IC) para cada
    indicador técnico y cada horizonte temporal (60 / 90).

    Para cada indicador y horizonte se evalúa su relación con:
    - delta_pts_h        → magnitud del movimiento futuro (continuo)
    - trade_h            → dirección (-1, 0, +1)
    - trade_h_only       → dirección pura (excluye no-trade)
    - target_op_h        → evento operativo (0 / 1)
    - target_tail_h      → evento extremo (0 / 1)

    Parámetro clave:
    - use_daily_ic = True
        Calcula IC por día y luego promedia (más robusto estadísticamente)
    - use_daily_ic = False
        Calcula IC global usando todas las filas juntas
    """


In [63]:
ohlcv_cols = ["open", "high", "low", "close", "volume"]


ic_ohlcv = compute_ic_table(
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    horizons=(60, 90),
    window_start="08:10",
    window_end="08:50",
    use_daily_ic=True
)

ic_ohlcv

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta
0,close,60,-0.436612,-0.276980,-0.616736,-0.003479,0.041025,,0.436612
1,low,60,-0.410441,-0.261641,-0.598534,-0.004301,0.046592,,0.410441
2,high,60,-0.409593,-0.265936,-0.591822,-0.001546,0.047149,,0.409593
3,open,60,-0.379016,-0.248909,-0.569774,-0.005479,0.050401,,0.379016
4,volume,60,0.010913,-0.022487,-0.017030,-0.010307,-0.121569,,0.010913
5,close,90,-0.350861,-0.281368,-0.554601,0.000452,0.027412,,0.350861
6,low,90,-0.329593,-0.269673,-0.538775,-0.000669,0.019508,,0.329593
7,high,90,-0.327931,-0.264842,-0.540773,0.002770,0.027836,,0.327931
8,open,90,-0.302396,-0.250195,-0.517750,0.004124,0.025093,,0.302396
9,volume,90,0.015236,0.024933,0.070706,0.001875,-0.020775,,0.015236


#### **6.1.1. Análisis de resultados de IC**

**1. OHLC presentan señal fuerte y coherente**

Las variables de precio (`open`, `high`, `low`, `close`) exhiben valores de:

$$
|IC_{\Delta}| \approx 0.30 \;-\; 0.44
$$

con **signo negativo consistente** tanto en `IC_delta` como en `IC_trade_only`.

Esto indica que:

- El **nivel de precio en la ventana de gestación** contiene información relevante
  sobre la magnitud y dirección del movimiento futuro.
- El régimen dominante es de **reversión**, consistente con lo observado
  previamente en los indicadores técnicos.
- La señal es incluso **más intensa cuando hay trade**, dado que:

$$
|IC_{trade\_only}| > |IC_{\Delta}|
$$

En términos puramente estadísticos, las variables OHLC muestran una capacidad
predictiva **comparable en magnitud** a la de los mejores indicadores técnicos.

<br>

**2. El volumen no presenta señal directa relevante**

La variable `volume` muestra:

$$
|IC_{\Delta}| \approx 0.01 \;-\; 0.015
$$

con valores cercanos a cero y sin coherencia clara entre los distintos targets.

Esto indica que, en la ventana de gestación analizada:

- El volumen **no explica directamente** la magnitud del movimiento futuro.
- Su aporte como señal anticipatoria es **débil o nulo**.





In [105]:
ic_ohlcv_operation_window = compute_ic_table(
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    horizons=(60, 90),
    window_start="09:00",
    window_end="10:00",
    use_daily_ic=True
)

ic_ohlcv_operation_window

,indicator,horizon,IC_delta,IC_trade,IC_trade_only,IC_target_op,IC_target_tail,note,abs_IC_delta
0,close,60,-0.609561,-0.477419,-0.619896,-0.045748,-0.006955,,0.609561
1,low,60,-0.585194,-0.460370,-0.608963,-0.059394,-0.018242,,0.585194
2,high,60,-0.576429,-0.458927,-0.611193,-0.030020,0.004988,,0.576429
3,open,60,-0.550173,-0.440190,-0.601221,-0.042425,-0.004050,,0.550173
4,volume,60,0.014239,0.014280,0.045481,-0.001604,0.000016,,0.014239
5,close,90,-0.663969,-0.498145,-0.609049,-0.052378,-0.004357,,0.663969
6,low,90,-0.637814,-0.482658,-0.598884,-0.062287,-0.016394,,0.637814
7,high,90,-0.629957,-0.479513,-0.602977,-0.043055,0.005500,,0.629957
8,open,90,-0.601743,-0.462251,-0.594261,-0.049518,-0.005376,,0.601743
9,volume,90,0.009452,0.004755,0.021479,-0.030292,-0.026205,,0.009452


#### **6.1.2. Interpretación metodológica**

Aunque las variables OHLC presentan valores de IC elevados, este resultado
**no invalida** el rol central de los indicadores técnicos. Por el contrario,
confirma que:

- Los indicadores técnicos capturan y **reexpresan información ya contenida
  en el precio**, de forma más estructurada y filtrada.
- Las variables OHLC actúan como **variables de estado del mercado**, no como
  señales diseñadas explícitamente para anticipar movimientos.

En este contexto, el IC elevado de OHLC es esperable y no implica que deban
sustituir a los indicadores técnicos, sino que:

- **Refuerzan la consistencia del análisis**, al confirmar el régimen dominante.
- Aportan **contexto estructural** útil para el modelo.

#### **6.1.3. Decisión de diseño del modelo**


En base a este análisis, se adopta el siguiente criterio:

- Las variables **OHLC** se mantienen como **features base**, normalizadas o
  transformadas según corresponda.
- Los **indicadores técnicos seleccionados** constituyen el **núcleo predictivo**
  del modelo.
- La variable **`volume` no se utiliza como feature principal** en esta etapa,
  pudiendo reservarse para análisis contextuales futuros.

Esta decisión permite equilibrar **información estructural**, **capacidad
predictiva real** e **interpretabilidad económica**, evitando tanto redundancia
innecesaria como exclusiones arbitrarias.

### **6.2. Correlación entre OHLCV e indicadores técnicos seleccionados**


El objetivo de este paso es evaluar redundancia informativa, es decir, determinar si las variables OHLCV ya están implícitamente capturadas por los indicadores técnicos seleccionados.

In [64]:
technical_indicators_finals

['price_ema60', 'momentum_10', 'roc_30', 'roc_60']

In [65]:
corr_ohlcv_vs_indicators = (
    mnq_intraday_with_indicators[
        ohlcv_cols + technical_indicators_finals
    ]
    .corr(method="spearman")
    .loc[ohlcv_cols, technical_indicators_finals]
)

corr_ohlcv_vs_indicators

,price_ema60,momentum_10,roc_30,roc_60
open,-0.011215,-0.005153,-0.007851,-0.007682
high,-0.011046,-0.004873,-0.007731,-0.007642
low,-0.010808,-0.004716,-0.007540,-0.007420
close,-0.010662,-0.004462,-0.007442,-0.007388
volume,-0.070137,-0.036393,-0.056235,-0.073238


#### **6.2.1. Análisis de Correlación entre variables OHLCV e indicadores técnicos seleccionados**

Se analizó la correlación de Spearman entre las variables OHLCV (`open`, `high`,
`low`, `close`, `volume`) y el conjunto final de indicadores técnicos seleccionados
(`price_ema60`, `momentum_10`, `roc_30`, `roc_60`), con el objetivo de evaluar
posible redundancia informativa.

**1. OHLC y los indicadores técnicos no son redundantes**

Las variables de precio (`open`, `high`, `low`, `close`) presentan correlaciones
muy cercanas a cero respecto a todos los indicadores técnicos:

$$
|\rho| \approx 0.00 \;-\; 0.01
$$

Esto indica que:

- Las variables OHLC **no están linealmente correlacionadas** con los indicadores.
- No existe colinealidad directa entre el estado del precio y las señales técnicas.
- Aunque ambos grupos tengan IC elevado respecto al target, **aportan información
  desde perspectivas distintas**.

<br>

**2. El volumen muestra correlación débil y no estructural**

La variable `volume` presenta correlaciones ligeramente mayores en magnitud:

$$
|\rho| \approx 0.03 \;-\; 0.07
$$

Aun así:

- Las correlaciones siguen siendo bajas.
- No indican dependencia fuerte con los indicadores técnicos.
- En combinación con su bajo IC, se refuerza que el volumen **no aporta señal
  predictiva directa** en esta etapa.

#### **6.2.2. Conclusión metodológica**


El análisis cruzado confirma que:

- Las variables **OHLC** aportan información de **estado del mercado**.
- Los **indicadores técnicos** capturan **estructura, dinámica y régimen**.
- Ambos conjuntos son **complementarios**, no redundantes.

Por lo tanto, se justifica incluir OHLC e indicadores técnicos seleccionados como
features del modelo, sin riesgo de colinealidad ni duplicación innecesaria de
información.


## **7. Consolidación de features**


A partir del análisis estadístico y estructural realizado en este stage, se concluye
que el conjunto de features adecuado para el entrenamiento del modelo está compuesto por:

- **Variables OHLC**: `open`, `high`, `low`, `close`
- **Indicadores técnicos seleccionados**:
  - `price_ema60`
  - `momentum_10`
  - `roc_30`
  - `roc_60`



In [66]:
technical_indicators_finals = ["price_ema60", "momentum_10",  "roc_30", "roc_60"]

### **7.1. Justificación**


1. Fuerza predictiva comprobada

    Tanto las variables OHLC como los indicadores técnicos seleccionados presentan valores de Information Coefficient elevados frente a los targets definidos, con régimen direccional consistente en los horizontes de 60 y 90 minutos.

2. Coherencia direccional

    Los indicadores técnicos seleccionados muestran coherencia total entre IC_delta y IC_trade_only, confirmando estabilidad de régimen e interpretación económica clara.

3. Ausencia de redundancia crítica

    El análisis de correlación cruzada demuestra que:

    - Las variables OHLC no están linealmente correlacionadas con los indicadores técnicos seleccionados.
    - Los indicadores técnicos capturan información estructural y dinámica que no es una reexpresión directa del precio crudo.

4. Complementariedad informativa

    - OHLC aporta el estado del mercado.
    - Los indicadores técnicos aportan estructura, momentum y régimen.

    Ambos conjuntos son complementarios y necesarios para describir el proceso generador del movimiento intradía.

### **7.2. Decisión final y set consolidado de features**


A partir del análisis estadístico y estructural realizado, se decide entrenar el modelo utilizando exclusivamente el siguiente conjunto de features:

```
finals_features = {
    "open", "high", "low", "close",
    "price_ema60",
    "momentum_10",
    "roc_30", "roc_60",
}
```
Este set constituye un conjunto consolidado, válido tanto para horizontes de 60 como de 90 minutos, dado que las diferencias observadas entre ambos son cuantitativas y no cualitativas.

| Tipo de feature            | Variable                            | Rol económico principal                          |
|----------------------------|-------------------------------------|--------------------------------------------------|
| Precio intradía            | `open`, `high`, `low`, `close`      | Estado instantáneo y rango del precio            |
| Tendencia de precio        | `price_ema60`                       | Nivel tendencial y extensión del movimiento      |
| Momentum de extensión      | `momentum_10`                       | Intensidad del desplazamiento reciente           |
| Momentum direccional       | `roc_30`, `roc_60`                  | Velocidad y dirección del movimiento             |

Este conjunto combina información estructural (precio) con señales dinámicas (tendencia y momentum), ofreciendo un equilibrio óptimo entre capacidad predictiva, interpretabilidad económica y robustez estadística, y evitando tanto la redundancia innecesaria como la inclusión de ruido no informativo.

Nota: el componente de momentum direccional puede adaptarse al horizonte de predicción (`roc_60` para 60 min y `roc_30` para 90 min) sin modificar la arquitectura ni la lógica económica del modelo.

### **7.4. Implicancia para el pipeline**


Este set consolidado constituye el núcleo de features técnicas del modelo y será utilizado en las etapas posteriores para:

- entrenamiento y validación del modelo predictivo,
- análisis de contribución por feature,
- y evaluación de desempeño económico.

La consolidación reduce la dimensionalidad, mejora la interpretabilidad y alinea el diseño del modelo con la estructura temporal y económica observada en los datos.

### **4.3. Código para calcular indicadores técnicos consolidados**


In [67]:
mnq_intraday_labeled = load_data()

In [68]:
mnq_intraday_labeled

,date,open,high,low,close,volume,delta_pts_60,trade_60,target_op_60,target_tail_60,delta_pts_90,trade_90,target_op_90,target_tail_90
datetime,,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,9.00,0,0,0,6.00,0,0,0
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,9.25,0,0,0,7.50,0,0,0
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,9.75,0,0,0,8.00,0,0,0
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,8.50,0,0,0,8.00,0,0,0
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,8.00,0,0,0,7.75,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,3251,NaN,0,0,0,NaN,0,0,0
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,2201,NaN,0,0,0,NaN,0,0,0
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,1859,NaN,0,0,0,NaN,0,0,0


In [69]:
from typing import List, Tuple
import pandas as pd
import ta
from ta.volatility import BollingerBands
from ta.momentum import ROCIndicator


def compute_selected_indicators_per_day(
    df: pd.DataFrame,
    target_col: str = "close",
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Computes selected technical indicators WITHOUT crossing days.
    All indicators are calculated independently per trading day
    using groupby('date').

    Indicators computed:
      - price_ema60
      - bb_60_15
      - rsi_14
      - roc_30
      - roc_60
      - macd

    Returns
    -------
    df_out : pd.DataFrame
        DataFrame with technical indicators added.
    indicator_columns : list[str]
        List of generated indicator column names.
    """

    indicator_columns: List[str] = [
        "price_ema60",
        "momentum_10",
        "roc_30",
        "roc_60",
          ]

    def apply_per_day(day_df: pd.DataFrame) -> pd.DataFrame:
        day_df = day_df.copy()

        # EMA-based price extension (normalized)
        day_df["price_ema60"] = (
            day_df[target_col] / day_df[target_col].ewm(span=60).mean() - 1
        )

        # Momentum (percentage change)
        day_df["momentum_10"] = day_df[target_col].pct_change(10)

        # Rate of Change (30, 60)
        day_df["roc_30"] = ROCIndicator(
            close=day_df[target_col],
            window=30
        ).roc()

        day_df["roc_60"] = ROCIndicator(
            close=day_df[target_col],
            window=60
        ).roc()


        return day_df

    df_out = df.groupby("date", group_keys=False).apply(apply_per_day)

    return df_out, indicator_columns



In [70]:
mnq_technical_indicators, technical_indicators_features = compute_selected_indicators_per_day(mnq_intraday_labeled)

In [71]:
ohlc_features = ['open', 'high', 'low', 'close']
final_features = ohlc_features + technical_indicators_finals
final_targets = ['delta_pts_60', 'delta_pts_90']
auxiliary_col = ['date']

In [72]:
selected_columns = auxiliary_col + final_features + final_targets
mnq_features_targets = mnq_technical_indicators[selected_columns].copy()

In [73]:
mnq_features_targets

,date,open,high,low,close,price_ema60,momentum_10,roc_30,roc_60,delta_pts_60,delta_pts_90
datetime,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,0.000000,NaN,NaN,NaN,9.00,6.00
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,-0.000070,NaN,NaN,NaN,9.25,7.50
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,-0.000140,NaN,NaN,NaN,9.75,8.00
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,-0.000040,NaN,NaN,NaN,8.50,8.00
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,-0.000031,NaN,NaN,NaN,8.00,7.75
...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 15:56:00-04:00,2025-06-13,21624.50,21635.00,21613.50,21617.50,-0.001396,0.000278,-0.055480,-0.203125,NaN,NaN
2025-06-13 15:57:00-04:00,2025-06-13,21616.50,21635.25,21615.75,21623.75,-0.001071,0.000243,-0.025429,-0.182336,NaN,NaN
2025-06-13 15:58:00-04:00,2025-06-13,21623.25,21632.75,21616.50,21621.75,-0.001126,0.000104,-0.024275,-0.102800,NaN,NaN


In [74]:
info_dataset(mnq_features_targets)

# Eliminar filas con al menos un NaN
mnq_features_targets = mnq_features_targets.dropna()

info_dataset(mnq_features_targets)


Información del dataset:

	Cantidad de días: 1303
	Registros por día: 571
	Hora diaria de inicio 06:30
	Hora diaria de final 16:00
	Zona horaria: America/New_York
Información del dataset:

	Cantidad de días: 1303
	Registros por día: 421
	Hora diaria de inicio 07:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York


In [75]:
OUT_PARQUET

PosixPath('/content/drive/MyDrive/neural_profit/data/features/mnq_features_target.parquet')

In [76]:
mnq_features_targets

,date,open,high,low,close,price_ema60,momentum_10,roc_30,roc_60,delta_pts_60,delta_pts_90
datetime,,,,,,,,,,,
2019-12-23 07:30:00-05:00,2019-12-23,8736.00,8736.75,8736.00,8736.75,0.000510,0.000143,0.100252,0.103119,0.25,0.50
2019-12-23 07:31:00-05:00,2019-12-23,8736.00,8736.00,8735.75,8735.75,0.000381,0.000143,0.100264,0.105999,0.75,2.25
2019-12-23 07:32:00-05:00,2019-12-23,8735.50,8735.50,8734.50,8735.00,0.000284,0.000086,0.080202,0.111745,1.00,3.00
2019-12-23 07:33:00-05:00,2019-12-23,8735.00,8735.75,8734.25,8734.50,0.000218,0.000086,0.071607,0.097410,2.25,3.25
2019-12-23 07:34:00-05:00,2019-12-23,8734.50,8734.50,8734.00,8734.00,0.000155,0.000029,0.060146,0.091680,3.75,3.50
...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,21716.50,21722.75,21712.00,21719.50,-0.001801,-0.000207,-0.309818,-0.357839,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,21719.00,21719.75,21695.75,21698.50,-0.002675,-0.000714,-0.406206,-0.419917,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,21698.25,21700.25,21670.50,21679.25,-0.003444,-0.001577,-0.488852,-0.493419,-52.25,-57.50


In [77]:
# Asegurar que exista el directorio
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Guardar dataset
mnq_features_targets.to_parquet(OUT_PARQUET, index=True)

## **8. Machine Learning for Algorithmic Trading**

### **8.1. Convertir indicadores en features estadísticamente estables**

Hasta ahora los tratamos como series crudas.

El siguiente nivel es normalizarlos en contexto intradía:

- Z-score rolling por día
- Percentil intradía
- Distancia al régimen típico del día

Ejemplo conceptual:
- roc_30 no vale por su valor absoluto
- vale por qué tan extremo es respecto a su distribución intradía

Esto aumenta IC out-of-sample sin cambiar indicadores.


In [78]:
final_features

['open',
 'high',
 'low',
 'close',
 'price_ema60',
 'momentum_10',
 'roc_30',
 'roc_60']

In [79]:
technical_indicators_finals #Lo que va a mostrar ['price_ema60', 'momentum_10', 'roc_30', 'roc_60']
final_targets  #Lo que va a mostrar ['delta_pts_60', 'delta_pts_90']

['delta_pts_60', 'delta_pts_90']

In [80]:
import numpy as np
import pandas as pd

# ============================================================
# Configuración (usa tus listas ya definidas)
# ============================================================
# technical_indicators_finals = ['price_ema60', 'momentum_10', 'roc_30', 'roc_60']
# final_targets = ['delta_pts_60', 'delta_pts_90']

EPS = 1e-12  # Para evitar división por cero en la std


# ============================================================
# Función: Z-score "point-in-time" por día (sin fuga)
# ============================================================
def add_expanding_zscore_by_day(
    df: pd.DataFrame,
    cols: list[str],
    date_col: str = "date",
    min_periods: int = 10,
) -> pd.DataFrame:
    """
    Crea features normalizadas por día usando estadísticos "hasta el momento".

    Para cada día (date_col), y para cada minuto t dentro de ese día:
        z(t) = (x(t) - mean(x[<=t])) / std(x[<=t])

    Ventajas:
    - No mezcla días (cada día se normaliza por separado)
    - No usa información del futuro (solo datos hasta t)
    - Suele mejorar estabilidad OOS cuando hay cambios de régimen/volatilidad

    Parámetros:
    - cols: columnas a normalizar
    - date_col: columna que identifica el día (por ejemplo, 'date')
    - min_periods: mínimos puntos del día para empezar a calcular mean/std
                  (antes de eso devuelve NaN en el z-score)
    """
    out = df.copy()

    # Validaciones mínimas
    if not isinstance(out.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser DatetimeIndex (datetime).")
    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' para agrupar por día.")

    # Asegura orden temporal global (por seguridad)
    out = out.sort_index()

    # Agrupa por día (no reordena los grupos)
    g = out.groupby(date_col, sort=False)

    # Calcula z-score expanding por día para cada feature
    for c in cols:
        exp_mean = (
            g[c]
            .expanding(min_periods=min_periods)
            .mean()
            .reset_index(level=0, drop=True)
        )
        exp_std = (
            g[c]
            .expanding(min_periods=min_periods)
            .std(ddof=0)
            .reset_index(level=0, drop=True)
        )

        # Feature normalizada
        out[f"{c}_z_exp"] = (out[c] - exp_mean) / (exp_std + EPS)

    return out


# ============================================================
# Pipeline: crea raw + z_exp para tus indicadores finales
# ============================================================
df = mnq_features_targets.copy()

# 1) Nos quedamos con las columnas necesarias (features + targets + date)
#    (esto evita arrastrar columnas que no usaremos)
keep_cols = ["date"] + technical_indicators_finals + [c for c in final_targets if c in df.columns]
df = df.loc[:, keep_cols].copy()

# 2) Creamos columnas *_raw explícitas (para comparar raw vs normalizado)
for c in technical_indicators_finals:
    df[f"{c}_raw"] = df[c]

# 3) Creamos columnas normalizadas *_z_exp (expanding z-score por día)
df = add_expanding_zscore_by_day(
    df,
    cols=technical_indicators_finals,
    date_col="date",
    min_periods=10,  # Ajustable: 5–15 suele ser razonable intradía
)

# 4) Armamos dataset final (raw + z_exp + targets)
final_features = (
    [f"{c}_raw" for c in technical_indicators_finals] +
    [f"{c}_z_exp" for c in technical_indicators_finals]
)

final_cols = ["date"] + final_features + [c for c in final_targets if c in df.columns]

# 5) Eliminamos filas donde aún no hay z-score (primeros min_periods-1 minutos de cada día)
mnq_features_targets_norm = df.loc[:, final_cols].dropna()

#print(mnq_features_targets_norm.head(3))


In [81]:
mnq_features_targets_norm

,date,price_ema60_raw,momentum_10_raw,roc_30_raw,roc_60_raw,price_ema60_z_exp,momentum_10_z_exp,roc_30_z_exp,roc_60_z_exp,delta_pts_60,delta_pts_90
datetime,,,,,,,,,,,
2019-12-23 07:39:00-05:00,2019-12-23,0.000126,-0.000229,0.002862,0.065878,-0.889593,-2.171343,-2.230969,-2.485505,5.75,3.00
2019-12-23 07:40:00-05:00,2019-12-23,0.000094,-0.000343,-0.065793,0.051551,-1.043967,-2.201886,-2.587049,-2.345790,6.50,2.50
2019-12-23 07:41:00-05:00,2019-12-23,0.000063,-0.000258,-0.031478,0.057284,-1.166773,-1.398822,-1.553999,-1.666255,8.00,3.00
2019-12-23 07:42:00-05:00,2019-12-23,0.000060,-0.000172,-0.034339,0.068748,-1.081699,-0.785472,-1.413645,-0.981199,8.25,1.00
2019-12-23 07:43:00-05:00,2019-12-23,-0.000024,-0.000200,-0.034342,0.057289,-1.538731,-0.908847,-1.274351,-1.403538,9.25,-1.50
...,...,...,...,...,...,...,...,...,...,...,...
2025-06-13 14:26:00-04:00,2025-06-13,-0.001801,-0.000207,-0.309818,-0.357839,-1.505576,-0.212917,-1.499326,-1.607481,-90.00,-102.00
2025-06-13 14:27:00-04:00,2025-06-13,-0.002675,-0.000714,-0.406206,-0.419917,-2.108045,-0.597544,-1.916900,-1.826768,-69.25,-74.75
2025-06-13 14:28:00-04:00,2025-06-13,-0.003444,-0.001577,-0.488852,-0.493419,-2.622274,-1.251375,-2.266248,-2.083547,-52.25,-57.50


In [82]:
import numpy as np
import pandas as pd

# ============================================================
# IC (Information Coefficient) IS vs OOS
# ------------------------------------------------------------
# - IC = correlación (Spearman por defecto) entre feature y target
# - In-sample (IS): rango de fechas para entrenamiento
# - Out-of-sample (OOS): rango posterior para evaluación
# - Devuelve un DataFrame con IC_IS, IC_OOS y delta (OOS-IS)
# ============================================================

def compute_ic_is_oos(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    date_col: str = "date",
    is_end_date: str = "2023-10-26",     # Ajuste a tu corte Train/Valid, ejemplo
    oos_start_date: str = "2023-10-27",  # Ajuste a tu inicio OOS, ejemplo
    method: str = "spearman",           # 'spearman' recomendado; 'pearson' si quieres lineal
    min_obs: int = 200,                 # mínimos registros para calcular IC confiable
) -> pd.DataFrame:
    """
    Calcula IC (feature vs target) por separado en In-Sample (IS) y Out-of-Sample (OOS).

    Parámetros:
    - df: DataFrame con index datetime y columnas features/target + date_col
    - features: lista de columnas feature a evaluar (ej: raw y z_exp)
    - target: columna objetivo (ej: 'delta_pts_60' o 'delta_pts_90')
    - date_col: columna con fecha diaria (tipo date o string 'YYYY-MM-DD')
    - is_end_date: última fecha incluida en IS (YYYY-MM-DD)
    - oos_start_date: primera fecha incluida en OOS (YYYY-MM-DD)
    - method: 'spearman' (robusto a outliers y no-lineal) o 'pearson'
    - min_obs: mínimo de filas requeridas para computar IC por bloque

    Retorna:
    - DataFrame con columnas:
        feature, n_is, ic_is, n_oos, ic_oos, delta_oos_minus_is
    """

    # Copia ligera
    data = df.copy()

    # Asegura que date_col sea comparable (datetime.date)
    if date_col not in data.columns:
        raise KeyError(f"Falta columna '{date_col}' en el DataFrame.")

    # Convierte a datetime (solo fecha) para filtrar por rango
    date_series = pd.to_datetime(data[date_col]).dt.date
    is_end = pd.to_datetime(is_end_date).date()
    oos_start = pd.to_datetime(oos_start_date).date()

    # Máscara IS/OOS (no se superponen)
    mask_is = date_series <= is_end
    mask_oos = date_series >= oos_start

    # Subsets
    df_is = data.loc[mask_is, :]
    df_oos = data.loc[mask_oos, :]

    if target not in data.columns:
        raise KeyError(f"Target '{target}' no existe en el DataFrame.")

    rows = []
    for f in features:
        if f not in data.columns:
            raise KeyError(f"Feature '{f}' no existe en el DataFrame.")

        # --- IS ---
        tmp_is = df_is[[f, target]].dropna()
        n_is = len(tmp_is)
        ic_is = np.nan
        if n_is >= min_obs:
            ic_is = tmp_is[f].corr(tmp_is[target], method=method)

        # --- OOS ---
        tmp_oos = df_oos[[f, target]].dropna()
        n_oos = len(tmp_oos)
        ic_oos = np.nan
        if n_oos >= min_obs:
            ic_oos = tmp_oos[f].corr(tmp_oos[target], method=method)

        rows.append({
            "feature": f,
            "n_is": n_is,
            "ic_is": ic_is,
            "n_oos": n_oos,
            "ic_oos": ic_oos,
            "delta_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_oos) and pd.notna(ic_is)) else np.nan
        })

    out = pd.DataFrame(rows).sort_values(by="ic_oos", ascending=False).reset_index(drop=True)
    return out





In [83]:
IN_SPLIT_ARTIFACT = Path(os.environ.get("IN_SPLIT_ARTIFACT", "reports/stage_05_time_aware_data_splitting_summary.json"))
IN_SPLIT_ARTIFACT = DRIVE_DIR / IN_SPLIT_ARTIFACT
# Carga del JSON
with IN_SPLIT_ARTIFACT.open("r") as f:
    stage_05_time_aware_data_splitting_summary = json.load(f)

In [84]:
from datetime import datetime

def extract_operational_dates(summary: dict) -> dict:
    splits = summary["details"]["splits"]

    def to_ymd(dt_str: str) -> str:
        return datetime.fromisoformat(dt_str).date().isoformat()

    return {
        "is_end_date": to_ymd(splits["train"]["datetime_max"]),
        "oos_start_date": to_ymd(splits["valid"]["datetime_min"]),
        "oos_end_date": to_ymd(splits["test"]["datetime_max"]),
    }

In [85]:
dates = extract_operational_dates(stage_05_time_aware_data_splitting_summary)
#dates['is_end_date']
#dates['oos_start_date']

In [86]:
features_to_test = [
    "price_ema60_raw", "price_ema60_z_exp",
    "momentum_10_raw", "momentum_10_z_exp",
    "roc_30_raw",      "roc_30_z_exp",
    "roc_60_raw",      "roc_60_z_exp",
 ]

In [87]:
final_targets

['delta_pts_60', 'delta_pts_90']

In [88]:
ic_60 = compute_ic_is_oos(
     df=mnq_features_targets_norm,
     features=features_to_test,
     target=final_targets[0],
     is_end_date=dates['is_end_date'],
     oos_start_date=dates['oos_start_date'],
     method="spearman",
     min_obs=200,
 )


In [89]:
ic_90 = compute_ic_is_oos(
     df=mnq_features_targets_norm,
     features=features_to_test,
     target=final_targets[1],
     is_end_date=dates['is_end_date'],
     oos_start_date=dates['oos_start_date'],
     method="spearman",
     min_obs=200,
 )

In [90]:
ic_60

,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,price_ema60_raw,375744,0.015478,161092,0.018679,0.003201
1,roc_60_raw,375744,0.014124,161092,0.014157,0.000034
2,roc_30_raw,375744,0.010452,161092,0.011878,0.001426
3,momentum_10_raw,375744,0.009099,161092,0.011030,0.001931
4,momentum_10_z_exp,375744,0.005656,161092,0.000488,-0.005168
5,roc_30_z_exp,375744,0.007936,161092,-0.000870,-0.008806
6,price_ema60_z_exp,375744,0.012730,161092,-0.002253,-0.014983
7,roc_60_z_exp,375744,0.011531,161092,-0.005283,-0.016815


In [91]:
ic_90

,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,price_ema60_raw,375744,0.017408,161092,0.015299,-0.002109
1,roc_60_raw,375744,0.016804,161092,0.014968,-0.001836
2,roc_30_raw,375744,0.014765,161092,0.008321,-0.006443
3,momentum_10_raw,375744,0.008853,161092,0.005255,-0.003598
4,momentum_10_z_exp,375744,0.004061,161092,-0.004046,-0.008106
5,roc_30_z_exp,375744,0.009233,161092,-0.005236,-0.014469
6,roc_60_z_exp,375744,0.012957,161092,-0.006333,-0.019290
7,price_ema60_z_exp,375744,0.011374,161092,-0.006634,-0.018008


#### **8.1.1. Observaciones y conclusiones**

1. Las versiones normalizadas (*_z_exp) no generalizan

    En ambos horizontes (H = 60 y H = 90):

    - Todas las features *_z_exp presentan:
      - IC positivo in-sample
      - IC cercano a cero o negativo out-of-sample
      - Un delta_oos_minus_is marcadamente negativo

    Conclusión: La normalización intradía mediante expanding z-score degradó la señal, en lugar de estabilizarla.


2. Las señales informativas están en los valores crudos

    Para ambos horizontes, las siguientes features:

    - `price_ema60_raw`  
    - `roc_60_raw`
    - `roc_30_raw`  
    - `momentum_10_raw`

    muestran:

    - IC positivo  
    - IC estable de in-sample a out-of-sample  
    - En H = 60, incluso mejora del IC OOS en algunos casos  

    Este es el comportamiento esperado de un factor robusto.


3. El horizonte H = 60 es más favorable que H = 90

    H = 60:
    - Varios factores mantienen o incrementan su IC out-of-sample  

    H = 90:
    - Todos los factores pierden algo de fuerza  
    - Aun así, mantienen IC positivo  

    Esto es consistente con un mercado intradiario de memoria corta, donde la señal se diluye a horizontes más largos.


4. Orden de importancia consistente entre horizontes

    Ranking por IC out-of-sample:

    H = 60:
    - `price_ema60_raw`
    - `roc_60_raw`  
    - `roc_30_raw`  
    - `momentum_10_raw`

    H = 90:
    - `price_ema60_raw`  
    - `roc_60_raw`  
    - `roc_30_raw`  
    - `momentum_10_raw`  

    Un ranking estable entre horizontes es señal de robustez estructural.

5. Decisión operativa

    - Eliminar las features *_z_exp  
    - Mantener únicamente las versiones raw  

**Diagnóstico final**

  Ya habíamos identificado los factores correctos.  
  Para la estructura intradía del MNQ, esta normalización no aporta valor adicional.

  Este resultado es positivo: evita complejidad innecesaria y confirma la solidez del proceso de selección previo.

### **8.2. Introducir interacciones mínimas**


No buscamos nuevos indicadores, sino capturar relaciones no lineales simples entre los cuatro factores ya validados, para que el modelo tenga acceso a información que no está en cada variable por separado.

Estas interacciones:
- no invalidan el análisis previo de IC,
- no introducen data snooping,
- suelen ser bien aprovechadas por modelos lineales y no lineales

Estas no son nuevos factores, son composiciones.

### **8.2.1. Interacciones recomendadas (mínimas y justificadas)**

#### 1. Diferencia de ROC: aceleración del movimiento

`roc_accel = roc_30 - roc_60`

Qué captura:
- Si el movimiento reciente es más fuerte que el de fondo → aceleración
- Si es más débil → desaceleración / agotamiento

Intuición
- No es tendencia (eso ya lo da EMA)
- Es cambio en la intensidad del momentum


#### 2. Momentum ponderado por estructura

`momentum_struct = momentum_10 × price_ema60_raw`

Qué captura
- Momentum alineado con la estructura dominante
- Penaliza momentum débil o contra-tendencia

Intuición
- El momentum “vale más” cuando ocurre dentro de una estructura clara

#### 3. Dirección x magnitud (desacople signo / tamaño)

`roc_signed = sign(momentum_10) × |roc_30|`

Qué captura
- Separación explícita entre:
- dirección (momentum)
- fuerza (ROC)

Intuición
- Dos movimientos con mismo ROC no son iguales si el momentum cambia de signo

### **8.2.2. Implementación**

In [92]:
df = mnq_features_targets.copy()

# 1) Aceleración del movimiento
df["roc_accel"] = df["roc_30"] - df["roc_60"]

# 2) Momentum ponderado por estructura
df["momentum_struct"] = df["momentum_10"] * df["price_ema60"]

# 3) Dirección × magnitud
df["roc_signed"] = np.sign(df["momentum_10"]) * np.abs(df["roc_30"])

In [93]:
interaction_features = [
    "roc_accel",
    "momentum_struct",
    "roc_signed",
]

In [94]:
ic_60_interactions = compute_ic_is_oos(
    df=df,
    features=interaction_features,
    target=final_targets[0],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)

In [95]:
ic_90_interactions = compute_ic_is_oos(
    df=df,
    features=interaction_features,
    target=final_targets[1],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)



### **8.2.3. Conclusiones — Análisis de IC para interacciones (IS vs OOS)**

In [96]:
ic_60_interactions

,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,momentum_struct,383952,0.006533,164611,0.016040,0.009507
1,roc_signed,383952,0.008390,164611,0.006823,-0.001567
2,roc_accel,383952,-0.013836,164611,-0.000845,0.012991


In [97]:
ic_90_interactions

,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,momentum_struct,383952,0.007142,164611,0.009112,0.001970
1,roc_signed,383952,0.009270,164611,0.000261,-0.009009
2,roc_accel,383952,-0.012432,164611,-0.007758,0.004674


1. `momentum_struct` **es válido y aporta señal**

    - **H = 60**
      - IC IS = 0.0065  
      - **IC OOS = 0.0160**  
      - Δ OOS-IS = **+0.0095**

    - **H = 90**
      - **IC OOS = 0.0091**

    **Interpretación**

    - La interacción *momentum x estructura*:
      - **mejora claramente out-of-sample**
      - generaliza mejor que varios factores base
    - Señal **real**, no atribuible a ruido

    Decisión: **Se mantiene**

2. `roc_signed` es **marginal / débil**

    - **H = 60**
      - IC OOS = 0.0068 (positivo pero bajo)
      - Δ OO-IS negativo

    - **H = 90**
      - IC OOS ≈ 0 (0.00026)
      - Pérdida marcada de señal OOS

    **Interpretación**

    - No es dañino, pero:
      - no agrega valor consistente
      - la señal se diluye al aumentar el horizonte

    Decisión: **Descartable**, o dejar solo para pruebas exploratorias

3. `roc_accel` **no aporta señal**

    - IC **negativo** tanto in-sample como out-of-sample  
    - No se observa relación explotable con el target

    Decisión: **Descartar sin dudar**

4. Comparación contra factor base  `momentum_struct` vs `momentum_10`


In [98]:
momentum_features = [
    "momentum_10",
    "momentum_struct",
    ]

ic_60_momentum = compute_ic_is_oos(
    df=df,
    features=momentum_features,
    target=final_targets[0],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)

ic_90_momentum = compute_ic_is_oos(
    df=df,
    features=momentum_features,
    target=final_targets[1],
    is_end_date=dates['is_end_date'],
    oos_start_date=dates['oos_start_date'],
    method="spearman",
    min_obs=200,
)

In [99]:
ic_60_momentum

,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,momentum_struct,383952,0.006533,164611,0.01604,0.009507
1,momentum_10,383952,0.008556,164611,0.01057,0.002014


In [100]:
ic_90_momentum

,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,momentum_struct,383952,0.007142,164611,0.009112,0.001970
1,momentum_10,383952,0.008472,164611,0.004544,-0.003928


- **`momentum_struct`**
  - Generaliza mejor que `momentum_10`
  - IC OOS **alto y estable** en **H=60** y **H=90**
  - Señal robusta fuera de muestra

- **`momentum_10`**
  - IC OOS menor e **inestable**
  - Se degrada claramente al pasar de **H=60 → H=90**

En conclusión:
- El **momentum alineado con estructura** es superior al momentum puro.
- **Decisión**: usar `momentum_struct` como feature principal; `momentum_10` es prescindible.

5. Decisión final de features

**Features base**
- `price_ema60`
- `roc_60`
- `roc_30`
- `momentum_10` (Se mantiene para calcular `momentum_struct`)

**Interacción aceptada**
- `momentum_struct`

**Features descartadas**

- `*_z_exp`
- `roc_signed`
- `roc_accel`



In [101]:
# 2) Momentum ponderado por estructura
df["momentum_struct"] = df["momentum_10"] * df["price_ema60"]

Diagnóstico final

> La señal intradía del MNQ **vive en los valores crudos**  
> y **se potencia cuando el momentum está alineado con la estructura**

### **8.3. Verificar estabilidad por horizonte**


#### **8.3.1. Introducción**


En predicción intradía, **un mismo indicador no necesariamente funciona igual para distintos horizontes de predicción**.  
Un feature puede capturar dinámicas de corto plazo (por ejemplo, impulsos rápidos), pero perder relevancia cuando el horizonte se extiende, o viceversa.

Por este motivo, no basta con que un indicador tenga **IC positivo**:  es necesario evaluar **si su relación con el target es estable cuando cambia el horizonte** \(H\).

Este análisis permite distinguir entre:
- señales **estructurales** del mercado, y
- señales **dependientes del horizonte** o directamente inestables.

El objetivo es que para cada feature, se busca determinar si:

- **Generaliza entre horizontes** (H = 60 y H = 90),
- Es **específica de un horizonte**, o
- Es **inestable** y debe descartarse.

En esta etapa **no se maximiza el IC**, sino que se evalúa **consistencia out-of-sample**.

**Criterios de clasificación**

Para cada feature \(f\):

1. Feature **estable**
    - IC OOS positivo en **H = 60 y H = 90**
    - Mantiene el signo
    - La variación de magnitud al aumentar H es moderada y explicable
    > Puede usarse en ambos horizontes.

2. Feature **especializada**
    - IC OOS positivo solo en **un horizonte**
    - IC OOS débil o cercano a cero en el otro
    - Sin comportamiento errático

    > Se utiliza únicamente en el horizonte donde aporta señal.

3. Feature **inestable**
    - IC OOS negativo
    - Cambios de signo sin patrón
    - Ruptura clara al variar el horizonte

    > Se descarta.

**Intuición clave**

- Un feature **estable** refleja una dinámica persistente del mercado.
- Un feature **especializado** captura efectos de escala temporal específica.
- Un feature **inestable** suele ser ruido o sobreajuste.

Este enfoque evita forzar indicadores fuera de su dominio natural de validez.

**Resultado esperado**

- Clasificar los features en lugar de eliminarlos arbitrariamente.
- Definir un **set coherente por horizonte**.
- Preparar el terreno para modelos que generalicen mejor out-of-sample.


#### **8.3.2. Implementación**


In [102]:
import numpy as np
import pandas as pd

# ============================================================
# Estabilidad por horizonte (H=60 vs H=90) usando IC IS/OOS
# ------------------------------------------------------------
# Requiere:
# - mnq_features_targets_norm: DataFrame con columnas features y targets
# - compute_ic_is_oos(df, features, target, ...) ya definida
# ============================================================

def evaluate_feature_stability_by_horizon(
    df: pd.DataFrame,
    features: list[str],
    target_h60: str = final_targets[0],
    target_h90: str = final_targets[1],
    date_col: str = "date",
    is_end_date: str = dates['is_end_date'],
    oos_start_date: str = dates['oos_start_date'],
    method: str = "spearman",
    min_obs: int = 200,
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Calcula IC IS/OOS para H=60 y H=90 y clasifica cada feature como:
    - stable:    IC OOS > 0 en ambos horizontes y signos consistentes
    - specialized_h60: aporta en H=60 pero no en H=90
    - specialized_h90: aporta en H=90 pero no en H=60
    - unstable:  IC OOS <= 0 en ambos o comportamiento errático

    Retorna un DataFrame con:
    - ic_is_60, ic_oos_60, delta_60
    - ic_is_90, ic_oos_90, delta_90
    - ratio_oos_90_over_60 (magnitud relativa)
    - stability_label
    """
    # IC por horizonte
    ic60 = compute_ic_is_oos(
        df=df,
        features=features,
        target=target_h60,
        date_col=date_col,
        is_end_date=is_end_date,
        oos_start_date=oos_start_date,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_60",
        "ic_oos": "ic_oos_60",
        "delta_oos_minus_is": "delta_60",
        "n_is": "n_is_60",
        "n_oos": "n_oos_60",
    })

    ic90 = compute_ic_is_oos(
        df=df,
        features=features,
        target=target_h90,
        date_col=date_col,
        is_end_date=is_end_date,
        oos_start_date=oos_start_date,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_90",
        "ic_oos": "ic_oos_90",
        "delta_oos_minus_is": "delta_90",
        "n_is": "n_is_90",
        "n_oos": "n_oos_90",
    })

    # Merge por feature
    out = ic60.merge(ic90, on="feature", how="inner")

    # Razón de magnitudes (cuánto queda de la señal al pasar de 60 -> 90)
    out["ratio_oos_90_over_60"] = (
        out["ic_oos_90"].abs() / (out["ic_oos_60"].abs() + eps)
    )

    # Etiquetado de estabilidad por horizonte
    def _label(row) -> str:
        o60 = row["ic_oos_60"]
        o90 = row["ic_oos_90"]

        # Si falta info OOS en alguno, marcar como unknown
        if pd.isna(o60) or pd.isna(o90):
            return "unknown"

        pos60 = o60 > 0
        pos90 = o90 > 0

        # Estable: positivo en ambos horizontes
        if pos60 and pos90:
            return "stable"

        # Especializado: solo uno aporta
        if pos60 and not pos90:
            return "specialized_h60"
        if pos90 and not pos60:
            return "specialized_h90"

        # Inestable: negativo o ~0 en ambos (según criterio de signo)
        return "unstable"

    out["stability_label"] = out.apply(_label, axis=1)

    # Orden sugerido: primero lo estable (y por IC OOS promedio)
    out["ic_oos_mean"] = out[["ic_oos_60", "ic_oos_90"]].mean(axis=1)
    out = out.sort_values(
        by=["stability_label", "ic_oos_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)

    # Columnas finales (ordenadas)
    cols = [
        "feature",
        "stability_label",
        "ic_is_60", "ic_oos_60", "delta_60",
        "ic_is_90", "ic_oos_90", "delta_90",
        "ratio_oos_90_over_60",
        "n_is_60", "n_oos_60",
        "n_is_90", "n_oos_90",
    ]
    return out[cols]


# ============================================================
# EJEMPLO DE USO
# ============================================================
# Lista de features a evaluar (ajustá a tu set final)
# Ejemplo: factores base + interacción validada
features_to_check = [
    "price_ema60",
    "roc_60",
    "roc_30",
    "momentum_10",
    "momentum_struct",
]

stability_report = evaluate_feature_stability_by_horizon(
    df=df,
    features=features_to_check,
    target_h60="delta_pts_60",
    target_h90="delta_pts_90",
    date_col="date",
    is_end_date="2023-10-26",
    oos_start_date="2023-10-27",
    method="spearman",
    min_obs=200,
)

#### **8.3.3.  Conclusiones — Estabilidad por horizonte y decisión sobre momentum**


In [103]:
stability_report

,feature,stability_label,ic_is_60,ic_oos_60,delta_60,ic_is_90,ic_oos_90,delta_90,ratio_oos_90_over_60,n_is_60,n_oos_60,n_is_90,n_oos_90
0,price_ema60,stable,0.015034,0.018227,0.003193,0.017230,0.014709,-0.002521,0.806986,383952,164611,383952,164611
1,roc_60,stable,0.013820,0.014218,0.000399,0.016738,0.014958,-0.001780,1.051998,383952,164611,383952,164611
2,momentum_struct,stable,0.006533,0.016040,0.009507,0.007142,0.009112,0.001970,0.568081,383952,164611,383952,164611
3,roc_30,stable,0.009941,0.011519,0.001578,0.014670,0.007986,-0.006684,0.693327,383952,164611,383952,164611
4,momentum_10,stable,0.008556,0.010570,0.002014,0.008472,0.004544,-0.003928,0.429904,383952,164611,383952,164611


1. Todos los features resultan **estables** (criterio formal)

    - IC OOS **positivo en H = 60 y H = 90**
    - Sin cambio de signo
    - No se detectan features inestables

      > El set es **coherente y consistente entre horizontes**.

2. Las diferencias aparecen en la **calidad de la estabilidad**

    No todos los features estables aportan el mismo valor.

    **Más robustos (mejor balance H=60 → H=90):**
    - `price_ema60`
    - `roc_60`

    Ambos mantienen:
    - IC OOS relativamente alto
    - Ratios cercanos a 1, indicando **persistencia de señal**

3. `momentum_struct` es **estable y valioso**

    - Muy fuerte en **H = 60**
    - En **H = 90**:
      - El IC OOS disminuye, pero
      - se mantiene **claramente positivo**
      - sin comportamiento errático

      > Feature estable, con **mayor relevancia en horizontes más cortos**.

4. `roc_30` y `momentum_10` son **estables pero más frágiles**

    - Caída clara del IC OOS al pasar de H = 60 a H = 90
    - Ratios bajos (≈ 0.4 – 0.7)

      > Aportan señal, pero de forma **secundaria**, especialmente en H = 90.

5. Justificación: por qué usar `momentum_struct` y no `momentum_10`

    Aunque `momentum_10` es formalmente estable, su señal:

    - es **más débil out-of-sample**,
    - se **degrada claramente** al aumentar el horizonte,
    - y queda **contenida** dentro de `momentum_struct`.

    `momentum_struct` combina:
    - dirección de corto plazo (momentum),
    - con alineación estructural (tendencia / contexto),

    lo que produce:
    - **mejor generalización OOS**,
    - mayor estabilidad entre horizontes,
    - y menor redundancia informativa.

    > Por simplicidad, robustez y poder explicativo,  
    **se utiliza `momentum_struct` como único feature de momentum**.

6. Decisión final de features

    **Core multi-horizonte**
    - `price_ema60`
    - `roc_60`

    **Refuerzo (especialmente H = 60)**
    - `momentum_struct`

    **Secundario**
    - `roc_30`

    **Descartado**
    - `momentum_10`

---

### Diagnóstico final

> El modelo intradía del MNQ se beneficia de
> **estructura + momentum alineado**, no de momentum aislado.

Con este criterio, el feature set queda **cerrado, parsimonioso y validado out-of-sample**,  listo para pasar a la etapa de **modelado**.

### **8.4. Separar señales de estructura, dirección y magnitud**

Después de seleccionar y validar features (IC IS/OOS + estabilidad por horizonte), organizamos el set final por **rol informativo**:

- **Estructura / régimen** (`price_ema60`): describe el “contexto” dominante del mercado.
- **Dirección alineada** (`momentum_struct`): captura el empuje direccional cuando está en sintonía con la estructura.
- **Magnitud / intensidad** (`roc_30`, `roc_60`): aproxima cuán grande puede ser el movimiento (escala/cambio).

**Por qué se hace:**
1. **Orden y claridad**: cada feature cumple un propósito distinto y evita ambigüedades.
2. **Mejor inductive bias**: el modelo aprende “dónde está el mercado”, “hacia dónde empuja” y “cuánto podría moverse”.
3. **Mejor interpretabilidad**: permite diagnosticar rápidamente qué tipo de señal está aportando (o no) cada grupo.

Este paso **no agrega indicadores**: solo formaliza lo que ya fue respaldado empíricamente.

In [104]:
import pandas as pd

# ============================================================
# Validación práctica del esquema "Estructura / Dirección / Magnitud"
# ------------------------------------------------------------
# Requiere:
# - mnq_features_targets_norm (DataFrame)
# - compute_ic_is_oos(...) ya definida
# ============================================================

# 1) Definir roles (ajusta nombres si tus columnas no tienen sufijos)
ROLE_MAP = {
    "structure":  ["price_ema60"],          # o "price_ema60_raw" si tu dataset usa raw
    "direction":  ["momentum_struct"],
    "magnitude":  ["roc_30", "roc_60"],     # idem: _raw si corresponde
}

# 2) Helper: calcula IC IS/OOS por feature y agrega columna "role"
def ic_by_role(
    df: pd.DataFrame,
    role_map: dict[str, list[str]],
    target: str,
    is_end_date: str,
    oos_start_date: str,
    method: str = "spearman",
    min_obs: int = 200,
    date_col: str = "date",
) -> pd.DataFrame:
    rows = []
    for role, feats in role_map.items():
        ic_tbl = compute_ic_is_oos(
            df=df,
            features=feats,
            target=target,
            date_col=date_col,
            is_end_date=is_end_date,
            oos_start_date=oos_start_date,
            method=method,
            min_obs=min_obs,
        ).copy()
        ic_tbl.insert(0, "role", role)
        rows.append(ic_tbl)

    out = pd.concat(rows, ignore_index=True)
    # Orden: por IC OOS descendente (lo que más importa)
    out = out.sort_values(["ic_oos"], ascending=False).reset_index(drop=True)
    return out


# 3) Ejecutar para H=60 y H=90 (ajusta fechas a tu split real)
IS_END = "2023-10-26"
OOS_START = "2023-10-27"

ic_roles_60 = ic_by_role(
    df=df,
    role_map=ROLE_MAP,
    target="delta_pts_60",
    is_end_date=IS_END,
    oos_start_date=OOS_START,
)

ic_roles_90 = ic_by_role(
    df=df,
    role_map=ROLE_MAP,
    target="delta_pts_90",
    is_end_date=IS_END,
    oos_start_date=OOS_START,
)

print("IC por rol (H=60):")
display(ic_roles_60)

print("\nIC por rol (H=90):")
display(ic_roles_90)


# 4) Validación adicional (simple): resumen por rol (promedios OOS)
def summarize_role_strength(ic_table: pd.DataFrame) -> pd.DataFrame:
    return (
        ic_table
        .groupby("role", as_index=False)
        .agg(
            n_features=("feature", "count"),
            ic_oos_mean=("ic_oos", "mean"),
            ic_oos_max=("ic_oos", "max"),
            ic_oos_min=("ic_oos", "min"),
        )
        .sort_values("ic_oos_mean", ascending=False)
        .reset_index(drop=True)
    )

role_summary_60 = summarize_role_strength(ic_roles_60)
role_summary_90 = summarize_role_strength(ic_roles_90)

print("\nResumen por rol (H=60):")
display(role_summary_60)

print("\nResumen por rol (H=90):")
display(role_summary_90)


IC por rol (H=60):


,role,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,structure,price_ema60,383952,0.015034,164611,0.018227,0.003193
1,direction,momentum_struct,383952,0.006533,164611,0.016040,0.009507
2,magnitude,roc_60,383952,0.013820,164611,0.014218,0.000399
3,magnitude,roc_30,383952,0.009941,164611,0.011519,0.001578



IC por rol (H=90):


,role,feature,n_is,ic_is,n_oos,ic_oos,delta_oos_minus_is
0,magnitude,roc_60,383952,0.016738,164611,0.014958,-0.001780
1,structure,price_ema60,383952,0.017230,164611,0.014709,-0.002521
2,direction,momentum_struct,383952,0.007142,164611,0.009112,0.001970
3,magnitude,roc_30,383952,0.014670,164611,0.007986,-0.006684



Resumen por rol (H=60):


,role,n_features,ic_oos_mean,ic_oos_max,ic_oos_min
0,structure,1,0.018227,0.018227,0.018227
1,direction,1,0.016040,0.016040,0.016040
2,magnitude,2,0.012869,0.014218,0.011519



Resumen por rol (H=90):


,role,n_features,ic_oos_mean,ic_oos_max,ic_oos_min
0,structure,1,0.014709,0.014709,0.014709
1,magnitude,2,0.011472,0.014958,0.007986
2,direction,1,0.009112,0.009112,0.009112


**H = 60:**
  
  - Estructura > Dirección > Magnitud
  - El momentum alineado (momentum_struct) es casi tan relevante como la estructura.

**H = 90:**

- Estructura ≈ Magnitud > Dirección
- La señal direccional pierde peso; dominan estructura y escala.

**Lectura final:**

A corto plazo manda la dirección alineada con estructura; al extender el horizonte, prevalece el contexto estructural y la magnitud del movimiento.

Este punto queda correctamente alineado con el Capítulo 4 (Feature Engineering / Alpha Factors).

Por qué:
- Seleccionó factores con IC IS/OOS y control de correlación.
- Validó generalización por horizonte.
- Refinó interacciones mínimas con respaldo OOS.
- Organizó el set por rol informativo (estructura, dirección, magnitud).

Cerró un feature set parsimonioso y robusto, listo para modelado.

Conclusión:

La notebook de feature engineering ya refleja fielmente el enfoque metodológico del capítulo: menos factores, mejor justificados, validados fuera de muestra.
